In [ ]:
import os
import glob
import json
import re
import subprocess
import shutil
import hashlib
import copy
import numpy as np
import logging
import javalang.tree as ast
import javalang.parse as parse
from tqdm import tqdm
from tree_sitter import Parser

from data.configuration import JAVA_LANGUAGE
from typing import List

In [ ]:
from data.configuration import d4j_proj_base, code_base
from utils.exceptions import *
from utils.java_parser import parse_methods_from_class_node

In [ ]:
import_json_path = "/path/to/repo/data/import_v2.json"
with open(import_json_path, "r") as file:
    import_json = json.load(file)
with open(os.path.join(code_base, "/path/to/repo/data/test_src.json"), "r") as f:
    content_path = json.load(f)

In [ ]:
# Retrieve the current PATH
current_path = os.environ.get('PATH')
print(f"Current PATH: {current_path}")

# Define the new directory you want to add
new_directory = '/path/to/defects4j/framework/bin'

# Update the PATH environment variable
os.environ['PATH'] = f"{new_directory}:{current_path}"

# Verify the update
updated_path = os.environ.get('PATH')
print(f"Updated PATH: {updated_path}")


In [ ]:
tgt_model = 'GPT-O3-MINI'
format = 'comment'
strategy = 'generation'
ablation = 'full'
input_version = 'buggy'
fixed_version = 'fixed'
buggy_version = 'buggy'
min_test_num = 10

In [ ]:

# file_post_fix = f'unified_invoked'
# file_post_fix = f'unified_invoked_docstring_prompt'
# file_post_fix = f'unified_invoked_docstring_cot_prompt'
# file_post_fix = f'unified_invoked_docstring_code_prompt'
# file_post_fix = f'unified_invoked_docstring_cot_prompt_min_test_num_{min_test_num}'
# file_post_fix = f'unified_invoked_docstring_cot_prompt_min_test_num_{min_test_num}_err_msg_prompt'
file_post_fix = f'unified_invoked_code_cot_few_shot_prompt_min_test_num_{min_test_num}'
# file_post_fix = f'unified_invoked_docstring_cot_few_shot_prompt_min_test_num_{min_test_num}'
# file_post_fix = f'unified_invoked_docstring_cot_few_shot_prompt'
# file_post_fix = f'unified_invoked_docstring_prompt_min_test_num_{min_test_num}'
# file_post_fix = f'unified_invoked_docstring_few_shot_prompt_min_test_num_{min_test_num}'
# file_post_fix = f'unified_invoked_generated_focal_prompt'
# file_post_fix = f'unified_invoked_generated_focal_prompt_min_test_num_{min_test_num}'
# file_post_fix = f'unified_invoked_generated_focal_err_msg_prompt_min_test_num_{min_test_num}'

In [ ]:
# Define the path to the JSONL file
jsonl_file_path = f'/path/to/repo/data/prompts/rq1/{tgt_model}_{format}_{strategy}_{ablation}_{input_version}_{file_post_fix}_with_completion.jsonl'
jsonl_file_path

In [ ]:
deprecated_bugs = ["Cli_6_", "Closure_63_", "Closure_93_", "JacksonDatabind_65_", "JacksonDatabind_89_", "Lang_2_", "Lang_18_", "Lang_25_", "Lang_48_", "Time_21_"]

In [ ]:
def is_method_public_or_protected(java_code):
    # Parse the Java code
    wrapped_java_code = 'class Test {\n' + java_code.replace('\r\n', '\n\n') + '\n}'
    tree = parse.parse(wrapped_java_code)

    # Search for the method
    for path, node in tree.filter(ast.MethodDeclaration):
        if node:
            if 'public' in node.modifiers or 'protected' in node.modifiers:
                return True
            return False
    raise ValueError(f"Method not found in the {java_code}")

In [ ]:
# Read the JSONL file and load the data
data = []
collected_id = set()
with open(jsonl_file_path, 'r') as f:
    for line in f:
        curr_json = json.loads(line)
        # check if curr_json['id'] starts with any of the deprecated_bugs
        if any(curr_json['id'].startswith(bug_id) for bug_id in deprecated_bugs):
            print(f"Skipping deprecated bug: {curr_json['id']}")
            continue
        if not is_method_public_or_protected(curr_json['focal_method']):
            print(f"Skipping private or default method: {curr_json['id']}")
            if "public" in curr_json['focal_method'] or "protected" in curr_json['focal_method']:
                print(f"Method: {curr_json['focal_method']}")
            continue
        collected_id.add(curr_json['id'])
        data.append(curr_json)

# Check if data is loaded
if not data:
    raise ValueError("No data found in the JSONL file.")

print(f"Total items in data: {len(data)}")


In [ ]:
print(len(set(collected_id)))

In [ ]:
example_item = data[0]

In [ ]:
print(example_item['content'])

In [ ]:
print(example_item['completion'])

In [ ]:
generated_test = example_item['completion']

In [ ]:
def parse_project_info(method_id):
    """
    Parses the method_id to extract project name, bug ID, and version.
    Expected format: projectName_bugId_version (e.g., Lang_46_fixed)
    """
    parts = method_id.split('_')
    if len(parts) < 3:
        raise ValueError(f"Invalid method_id format: {method_id}")
    project_name = parts[0]
    bug_id = parts[1]
    return project_name, bug_id


In [ ]:
# Parse project info
method_id = example_item['id']
project_name, bug_id = parse_project_info(method_id)
print(f"Project: {project_name}, Bug ID: {bug_id}, Test generated from version: {input_version}")

In [ ]:
# Set the base directory for the Defects4J project
base_checkout_dir = '/path/to/repo/d4j_proj_base'

# Create the directory if it doesn't exist
if not os.path.exists(base_checkout_dir):
    os.makedirs(base_checkout_dir)

# Define the project directory
project_dir_name = f"{project_name}_{bug_id}/{fixed_version}"
buggy_project_dir_name = f"{project_name}_{bug_id}/{buggy_version}"
project_dir = os.path.join(base_checkout_dir, project_dir_name)
buggy_project_dir = os.path.join(base_checkout_dir, buggy_project_dir_name)
print(f"Project directory: {project_dir}")
print(f"Bug directory: {buggy_project_dir}")

In [ ]:
def extract_package_name(method_signature):
    """
    Extracts the package name from the method signature.
    Expected format: package#ClassName#MethodName(parameters)
    """
    if not method_signature:
        raise ValueError("method_signature is empty")

    # Split the method_signature by '#' to separate package, class, and method
    parts = method_signature.split('#')
    if len(parts) < 2:
        raise ValueError(f"Invalid method_signature format: {method_signature}")

    package_name = parts[0]
    return package_name

In [ ]:
# Organize generated files in a structured way
# Create a directory to store generated tests and related files
generated_tests_dir = os.path.join(project_dir, 'generated_tests')
if not os.path.exists(generated_tests_dir):
    os.makedirs(generated_tests_dir)

# Revert to using a hashed package directory in `generated_tests`
method_signature = example_item.get('method_signature', '')
unique_hash = hashlib.md5(method_signature.encode('utf-8')).hexdigest()[:8]
hashed_package_name = f"testpackage_{unique_hash}"

# Create the directory for the test under `generated_tests` using the hashed name
test_package_dir = os.path.join(generated_tests_dir, hashed_package_name)
if not os.path.exists(test_package_dir):
    os.makedirs(test_package_dir)

# Save the generated test to a Markdown file in the test package directory
test_md_file_name = f"{method_id}_test.md"
test_md_file_path = os.path.join(test_package_dir, test_md_file_name)
with open(test_md_file_path, 'w') as f:
    f.write(generated_test)

print(f"\nGenerated unit test saved to {test_md_file_path}")

In [ ]:
from rq1.assistant_methods import extract_elements_from_llm_generation, analyze_method_signature_for_coverage
from utils.java_parser import parse_import_stmts_from_file_code, parse_method_invocation
import pickle
import javalang
import javalang.tree
import io
from rq1.assistant_methods import calculate_coverage_stats

In [ ]:
def assemble_empty_test_file(class_sig):
    """
    Assembles an empty test file.

    Returns:
        tuple: A tuple containing the new content of the test file and the fully qualified name of the test class.
    """
    package_name = ".".join(class_sig.split(".")[:-1])
    class_name = class_sig.split(".")[-1]
    new_content = f"package {package_name};\nimport org.junit.Test;\nimport static org.junit.Assert.assertTrue;\npublic class {class_name}EmptyTests{{\n    @Test\n    public void emptyTest(){{\n        assertTrue(true);\n    }}\n}}\n"
    return new_content, package_name + f".{class_name}EmptyTests"

In [ ]:
def filter_imports(src_imports: list, tgt_imports: set):
    """
    filter imports from source list from target list.
    :param src_imports: source import list, which is about to be merged.
    :param tgt_imports: target import list, which is the imports that generated by LLM
    :return: merged import set
    """

    # find all classes that are imported by target import statements
    classes_imported_by_tgt = []
    packages_imported_by_tgt = []
    final_imports = []
    jupiters_included_in_tgt = False
    for import_str in tgt_imports:
        tokens = import_str.split()
        if len(tokens) == 2 and tokens[0] == "import":
            cls_str = tokens[1].split(".")[-1][:-1]
            if "org.junit.jupiter" in tokens[1]:
                jupiters_included_in_tgt = True
            if cls_str == "*":
                packages_imported_by_tgt.append(".".join(tokens[1].split(".")[:-1]))
                pass
            else:
                classes_imported_by_tgt.append(cls_str)
            # final_imports.append(import_str)
            pass
        elif len(tokens) == 3 and tokens[0] == "import" and tokens[1] == "static":
            cls_str = tokens[2].split(".")[-1][:-1]
            if "org.junit.jupiter" in tokens[2]:
                jupiters_included_in_tgt = True
            if cls_str == "*":
                packages_imported_by_tgt.append(".".join(tokens[2].split(".")[:-1]))
                pass
            else:
                classes_imported_by_tgt.append(cls_str)
            # final_imports.append(import_str)
            pass
        else:
            raise NotImplementedError(
                f"more than 3 tokens in {import_str}, please check"
            )

    for import_str in src_imports:
        tokens = import_str.split()
        if len(tokens) == 2 and tokens[0] == "import":
            cls_str = tokens[1].split(".")[-1][:-1]
            imported_cls = tokens[1]
            pass
        elif len(tokens) == 3 and tokens[0] == "import" and tokens[1] == "static":
            cls_str = tokens[2].split(".")[-1][:-1]
            imported_cls = tokens[2]
            pass
        else:
            raise NotImplementedError(
                f"more than 3 tokens in {import_str}, please check"
            )

        if cls_str in classes_imported_by_tgt:
            # 同名的去掉
            continue

        # 去掉Assert 和 junit.jupiter.Assertion的问题
        # org.junit.jupiter.api.Assertions.*

        package_name = ".".join(imported_cls.split(".")[:-1])
        if (package_name in packages_imported_by_tgt) or (
                package_name.startswith("org.junit") and jupiters_included_in_tgt
        ):
            # 如果已经引入*，则不需要继续补充
            continue

        final_imports.append(import_str)

    return final_imports

In [ ]:
def extract_method_name(method_code):
    # Parse the Java code
    method_code = "public class TmpClass {\n" + method_code + "}\n"
    tree = javalang.parse.parse(method_code)

    method_names = []
    for _, node in tree.filter(javalang.tree.MethodDeclaration):
        method_names.append(node.name)
    return method_names[0]

In [ ]:
def is_static_class(class_content):
    if "static class" in class_content:
        return True
    else:
        return False
def assemble_test_class(
        package_name: str,
        imports,
        focal_class_import: list,
        src_imports: list,
        pre_defined_imports: list,
        class_declaration: str,
        new_methods: list,
        fields: list,
        classes: list,
):
    """
    Assembles a test class with the given parameters.

    Args:
        package_name (str): The name of the package.
        imports: The imports required for the test class.
        focal_class_import (list): The imports from the focal class.
        src_imports (list): The imports from the source.
        pre_defined_imports (list): The pre-defined imports.
        class_declaration (str): The declaration of the class.
        new_methods (list): The new methods to be added to the class.
        fields (list): The fields to be added to the class.
        classes (list): The additional classes to be added to the class.

    Returns:
        str: The assembled test class.
    """
    res_content = ""
    res_stream = io.StringIO(res_content)
    res_stream.write(f"package {package_name};\n")

    res_stream.write("// from focal class\n")
    for imp in focal_class_import:
        res_stream.write(imp + "\n")

    res_stream.write("\n// from src\n")
    for imp in src_imports:
        res_stream.write(imp + "\n")

    res_stream.write("\n// from LLM\n")
    for imp in imports:
        res_stream.write(imp + "\n")

    res_stream.write("\n// pre-defined\n")
    for imp in pre_defined_imports:
        res_stream.write(imp + "\n")

    res_stream.write("\n")
    res_stream.write(class_declaration + "\n")
    res_stream.write("\n")

    for field in fields:
        res_stream.write(field + "\n")

    for method in new_methods:
        res_stream.write("\n" + method + "\n")
        res_stream.write("\n")

    res_stream.write("}")

    for single_class in classes:
        if is_static_class(single_class):
            continue
        res_stream.write(f"\n{single_class}\n")

    assembled_test_class = res_stream.getvalue()
    return assembled_test_class

In [ ]:
def assemble_first_round_test_class(
        project_name,
        class_sig,
        imports,
        methods,
        fields,
        import_json,
        classes,
):
    """
    Assembles the first round test class.

    Args:
        project_name (str): The name of the project.
        class_sig (str): The signature of the class.
        imports (list): The list of imports.
        methods (list): The list of methods.
        fields (list): The list of fields.
        import_json (dict): The import JSON.
        classes (list): The list of classes.

    Returns:
        str: The assembled test class.
    """
    def _get_test_location_by_bug_id(bug_id):
        fixed_base = content_path[bug_id.lower()]["test"]
        buggy_base = content_path[bug_id.lower()]["test"]
        fixed_base = os.path.join(d4j_proj_base, bug_id, "fixed", fixed_base)
        buggy_base = os.path.join(d4j_proj_base, bug_id, "buggy", buggy_base)
        return fixed_base, buggy_base


    def _get_src_location_by_bug_id(bug_id):
        fixed_base = content_path[bug_id.lower()]["src"]
        buggy_base = content_path[bug_id.lower()]["src"][1:]
        fixed_base = os.path.join(d4j_proj_base, bug_id, "fixed", fixed_base)
        buggy_base = os.path.join(d4j_proj_base, bug_id, "buggy", buggy_base)
        return fixed_base, buggy_base
    focal_imports = [
        "import org.junit.Test;",
        "import org.junit.Assert;",
        "import org.junit.Before;",
        "import org.junit.After;",
        "import static org.junit.Assert.*;",
        "import org.junit.Ignore;",
        "import org.junit.BeforeClass;",
        "import org.junit.AfterClass;",
        "import org.junit.runner.RunWith;",
        "import org.junit.runners.JUnit4;",
        "import org.junit.Rule;",
        "import org.junit.rules.ExpectedException;",
        "import static org.mockito.Mockito.*;",
        "import org.mockito.Mockito;",
        "import static org.hamcrest.MatcherAssert.assertThat;",
        "import static org.hamcrest.Matchers.*;",
        "import java.text.SimpleDateFormat;",
        "import java.io.*;",
        "import java.lang.*;",
        "import java.util.*;",
        "import java.time.*;",
        "import java.math.*;",
        "import java.net.*;",
        "import java.security.*;",
        "import java.nio.file.Files;",
        "import java.nio.file.Path;",
    ]
    package_name = ".".join(class_sig.split(".")[:-1])
    focal_base_dir, _ = _get_src_location_by_bug_id(project_name)
    focal_class_file = class_sig.replace(".", "/") + ".java"
    focal_class_import = []
    if "Gson" in project_name or "Csv" in project_name:
        pass
    else:
        with open(
                os.path.join(focal_base_dir, focal_class_file),
                "r",
                encoding="iso8859-1",
        ) as f:
            focal_content = f.read()
        focal_class_import.extend(parse_import_stmts_from_file_code(focal_content))
    # print(focal_class_import)
    imports.append(f"import {class_sig};")
    # 根据LLM生成的 imports ，酌情加入imports
    # remove enum to avoid java version issue
    focal_class_import = [i for i in focal_class_import if "enum" not in i.split(".")]
    pre_defined_imports = [i for i in focal_imports if "enum" not in i.split(".")]
    import_list = pickle.loads(
        pickle.dumps(import_json[project_name]["fixed"])
    )  # 按src得到的imports
    src_imports = [i for i in import_list if "enum" not in i.split(".")]
    llm_imp_set = set(imports)
    focal_class_import = filter_imports(focal_class_import, llm_imp_set)
    pre_defined_imports = filter_imports(
        pre_defined_imports, set(focal_class_import) | llm_imp_set
    )
    src_imports = filter_imports(
        src_imports, llm_imp_set | set(focal_class_import) | set(pre_defined_imports)
    )

    class_declaration = f"public class {class_sig.split(".")[-1]}LLMGeneratedTests {{"

    current_method_names = {}
    new_methods = []
    for method in methods:
        try:
            method_name = extract_method_name(method)
            if method_name in current_method_names.keys():
                current_method_names[method_name] = (
                        current_method_names[method_name] + 1
                )
                method = method.replace(
                    method_name, method_name + str(current_method_names[method_name])
                )
            else:
                current_method_names[method_name] = 0
            new_methods.append(method)
        except:
            continue

    assembled_test_class = assemble_test_class(
        package_name,
        imports,
        focal_class_import,
        src_imports,
        pre_defined_imports,
        class_declaration,
        new_methods,
        fields,
        classes,
    )
    return assembled_test_class, package_name + f".{class_sig.split(".")[-1]}LLMGeneratedTests"

In [ ]:

def post_process_generation(data, index=0):

    """
    Post-process the generated test to clean up and organize the content.
    """
    res_dict = copy.deepcopy(data)
    llm_output = data["completion"]
    res_dict["output"] = llm_output
    res_dict["index"] = index
    bug_id = data["id"]  # Chart_10
    bug_id = "_".join(bug_id.split("_")[:2])
    res_dict["bug_id"] = bug_id
    focal_method = data["focal_method"]
    method_signature = data["method_signature"]

    extractions = extract_elements_from_llm_generation(
        llm_output, strategy, method_signature
    )
    (
        package_name,
        package_dir,
        class_name,
        clazz_dir,
        method_name,
        parameter_tuple,
    ) = analyze_method_signature_for_coverage(method_signature)
    run_res_dict = {}
    empty_test = False
    run_res_dict["num_uts"] = len(
        [m for m in extractions["methods"] if m.startswith("@Test")]
    )

    first_round_test_content = ""
    first_round_test_sig = None
    if extractions["msg"] == "success":
        # 收集到了足够的信息
        try:
            (
                first_round_test_content,
                first_round_test_sig,
            ) = assemble_first_round_test_class(
                bug_id,
                class_name,
                extractions["imports"],
                extractions["methods"],
                extractions["fields"],
                import_json,
                extractions["classes"],
            )
        except UnicodeDecodeError as e:
            first_round_test_content, first_round_test_sig = assemble_empty_test_file(class_name)
    else:
        first_round_test_content, first_round_test_sig = assemble_empty_test_file(class_name)
    return first_round_test_content, first_round_test_sig, extractions, (
        package_name,
        package_dir,
        class_name,
        clazz_dir,
        method_name,
        parameter_tuple,
    )

In [ ]:
print(post_process_generation(example_item)[0])

In [ ]:
# Extract Java code from the Markdown
code_blocks = re.findall(r'```java(.*?)```', generated_test, re.DOTALL)
if not code_blocks:
    # Try without specifying the language
    code_blocks = re.findall(r'```(.*?)```', generated_test, re.DOTALL)

# Save the Java code to a .java file
if code_blocks:
    java_code, test_sig, extractions, (
        package_name,
        package_dir,
        class_name,
        clazz_dir,
        method_name,
        parameter_tuple,
    ) = post_process_generation(example_item)
    # java_code = code_blocks[0].strip()
    print(java_code)

    # Count the number of test methods in the generated test
    # test_method_pattern = r'@Test\s*public\s+void\s+\w+\s*\('
    # test_methods = re.findall(test_method_pattern, java_code)
    # num_tests = len(test_methods)
    # print(f"Number of test methods found: {num_tests}")

    # Extract the class name from the Java code
    class_match = re.search(r'public\s+(?:class|interface)\s+(\w+)', java_code)
    if class_match:
        class_name_in_code = class_match.group(1)
        test_java_file_name = f"{class_name_in_code}.java"
    else:
        print("Could not find the class name in the Java code.")
        raise ValueError("Class name not found in the Java code.")

    # # Add package declaration to Java code
    # java_code_with_package = f"package {package_name};\n\n{java_code}"

    # Save the Java code with package declaration to the file in the test package directory
    test_java_file_path = os.path.join(test_package_dir, test_java_file_name)
    with open(test_java_file_path, 'w') as f:
        f.write(java_code)

    print(f"Unit test Java code saved to {test_java_file_path}\n")
else:
    print("No code block found in the generated test.\n")
    raise ValueError("No Java code found in the generated test.")


In [ ]:
# Use 'defects4j export' to get the test source directory
export_command = [
    'defects4j', 'export',
    '-p', 'dir.src.tests',
    '-w', project_dir
]
try:
    result = subprocess.run(export_command, check=True, stdout=subprocess.PIPE, text=True)
    test_src_rel_path = result.stdout.strip()
except subprocess.CalledProcessError as e:
    print(f"Failed to get test source directory: {e}")
    raise e

# print the test source directory
print(f"Test source directory: {test_src_rel_path}")

# Construct the absolute path to the test source directory
test_src_dir = os.path.join(project_dir, test_src_rel_path)

print(f"Test source directory: {test_src_dir}")

In [ ]:
# also export the buggy version
export_command = [
    'defects4j', 'export',
    '-p', 'dir.src.tests',
    '-w', buggy_project_dir
]
try:
    result = subprocess.run(export_command, check=True, stdout=subprocess.PIPE, text=True)
    buggy_test_src_rel_path = result.stdout.strip()
except subprocess.CalledProcessError as e:
    print(f"Failed to get buggy test source directory: {e}")
    raise e

# print the test source directory
print(f"Buggy test source directory: {buggy_test_src_rel_path}")

# Construct the absolute path to the test source directory
buggy_test_src_dir = os.path.join(buggy_project_dir, buggy_test_src_rel_path)

print(f"Buggy test source directory: {buggy_test_src_dir}")

In [ ]:
# Ensure the test source directory exists
if not os.path.exists(test_src_dir):
    print(f"Test source directory not found: {test_src_dir}")
    raise FileNotFoundError(f"Test source directory not found: {test_src_dir}")

# Construct the destination directory in the test source directory
method_signature = example_item.get('method_signature', '')
# package_name = extract_package_name(method_signature)

# Convert package name to directory path
# package_dir = package_name.replace('.', os.sep)
dst_test_package_dir = os.path.join(test_src_dir, package_dir)
if not os.path.exists(dst_test_package_dir):
    os.makedirs(dst_test_package_dir)

# Construct the destination Java file path
dst_java_file_path = os.path.join(dst_test_package_dir, test_java_file_name)
# Check if the test Java file already exists in the destination directory, and if so, remove it
if os.path.exists(dst_java_file_path):
    print(f"Removing existing test Java file: {dst_java_file_path}")
    os.remove(dst_java_file_path)

In [ ]:
# Ensure the buggy test source directory exists
if not os.path.exists(buggy_test_src_dir):
    print(f"Buggy test source directory not found: {buggy_test_src_dir}")
    raise FileNotFoundError(f"Buggy test source directory not found: {buggy_test_src_dir}")

# convert package name to the buggy test source directory
buggy_dst_test_package_dir = os.path.join(buggy_test_src_dir, package_dir)
if not os.path.exists(buggy_dst_test_package_dir):
    os.makedirs(buggy_dst_test_package_dir)

# Construct the destination Java file path
buggy_dst_java_file_path = os.path.join(buggy_dst_test_package_dir, test_java_file_name)
# Check if the test Java file already exists in the destination directory, and if so, remove it
if os.path.exists(buggy_dst_java_file_path):
    print(f"Removing existing test Java file: {buggy_dst_java_file_path}")
    os.remove(buggy_dst_java_file_path)

In [ ]:
# Do a test compilation of the project before adding the generated test
# Compile the project using Defects4J
compile_command = [
    'defects4j', 'compile',
    '-w', project_dir
]
print(f"Compiling project: {' '.join(compile_command)}")
pre_test_compile_log_file_path = os.path.join(test_package_dir, "pre_test_compile_log")
try:
    result = subprocess.run(
        compile_command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=True
    )
    # Print the captured output
    print("Defects4J output:\n" + result.stdout)
    # Save the log to a file
    with open(pre_test_compile_log_file_path, 'w') as f:
        f.write(result.stdout)
    # Check if the compilation was successful
    if result.returncode == 0:
        print("Project compiled successfully.")
    else:
        print("Project compilation failed.")
except subprocess.CalledProcessError as e:
    print(f"Compilation failed: {e}")
    # Save the log to a file
    with open(pre_test_compile_log_file_path, 'w') as f:
        f.write(e.stdout)
        # add a separator
        f.write("-" * 80 + "\n")
        # write return code
        f.write(f"Return code: {e.returncode}\n")
    print(f"error message: \n{e.stdout}")
    print(f"error return code: {e.returncode}")

In [ ]:
# Do a test compilation of the buggy project before adding the generated test
# Compile the project using Defects4J
compile_command = [
    'defects4j', 'compile',
    '-w', buggy_project_dir
]
print(f"Compiling buggy project: {' '.join(compile_command)}")
pre_test_compile_log_file_path = os.path.join(buggy_dst_test_package_dir, "pre_test_compile_log")
try:
    result = subprocess.run(
        compile_command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=True
    )
    # Print the captured output
    print("Defects4J output:\n" + result.stdout)
    # Save the log to a file
    with open(pre_test_compile_log_file_path, 'w') as f:
        f.write(result.stdout)
    # Check if the compilation was successful
    if result.returncode == 0:
        print("Buggy project compiled successfully.")
    else:
        print("Buggy project compilation failed.")
except subprocess.CalledProcessError as e:
    print(f"Buggy Compilation failed: {e}")
    # Save the log to a file
    with open(pre_test_compile_log_file_path, 'w') as f:
        f.write(e.stdout)
        # add a separator
        f.write("-" * 80 + "\n")
        # write return code
        f.write(f"Return code: {e.returncode}\n")
    print(f"error message: \n{e.stdout}")
    print(f"error return code: {e.returncode}")

In [ ]:

# Copy only the Java file to the destination directory
print(f"Copying {test_java_file_path} to {dst_java_file_path} and {buggy_dst_java_file_path}")
shutil.copyfile(test_java_file_path, dst_java_file_path)
shutil.copyfile(test_java_file_path, buggy_dst_java_file_path)


In [ ]:
from utils.dependency_analyzer import add_dependencies
from utils.cal_rate import compile_and_test

In [ ]:
# Add dependencies to the project
add_dependencies(d4j_proj_base, f"{project_name}_{bug_id}")

In [ ]:
res = compile_and_test(
    f"{project_name}_{bug_id}",
    test_sig,
    java_code,
    method_name,
    "class_cov_dir",
    "first",
    tgt_model,
    strategy,
    ablation,
    format,
)

In [ ]:
res["compiled"]

In [ ]:
num_tests = res['total_uts_num']

In [ ]:
new_res = copy.deepcopy(res)

In [ ]:
res["coverage_results"]

In [ ]:
new_res["compiled"]

In [ ]:
new_res["buggy_compiled"]

In [ ]:
def extract_failed_class_names(failed_imports):
    failed_classes = []
    for imported in failed_imports:
        # Remove the "import" keyword and strip leading/trailing whitespace and semicolon
        clean_import = imported.replace("import", "").strip().rstrip(";")

        # Split by dots to isolate the class name or wildcard
        parts = clean_import.split(".")

        # Get the last part only if it's not a wildcard import (*)
        if parts[-1] != "*":
            failed_classes.append(parts[-1])

    return failed_classes


In [ ]:
def assemble_recursive_test_classes(
        package_name,
        imports,
        methods,
        fields,
        classes,
        class_sig
):
    class_declaration = f"public class {class_sig.split(".")[-1]}LLMGeneratedTests {{"
    current_method_names = {}
    new_methods = []
    for method in methods:
        try:
            method_name = extract_method_name(method)
            if method_name in current_method_names.keys():
                current_method_names[method_name] = (
                        current_method_names[method_name] + 1
                )
                method = method.replace(
                    method_name, method_name + str(current_method_names[method_name])
                )
            else:
                current_method_names[method_name] = 0
            new_methods.append(method)
        except:
            continue

    assembled_test_class = assemble_test_class(
        package_name,
        [],
        [],
        [],
        imports,
        class_declaration,
        new_methods,
        fields,
        classes,
    )

    return assembled_test_class, package_name + f".{class_sig.split(".")[-1]}LLMGeneratedTests"

In [ ]:
num_round = 0
while not new_res["compiled"]:
    # clean up the non-compilable test
    if os.path.exists(dst_java_file_path):
        print(f"Removing test Java file after compilation failure: {dst_java_file_path}")
        os.remove(dst_java_file_path)

    # clean up the non-compilable test
    if os.path.exists(buggy_dst_java_file_path):
        print(f"Removing buggy test Java file after compilation failure: {buggy_dst_java_file_path}")
        os.remove(buggy_dst_java_file_path)

    if new_res["passed_uts_num"] == 0:
        empty_test, empty_test_sig = assemble_empty_test_file(class_name)
        empty_class_match = re.search(r'public\s+(?:class|interface)\s+(\w+)', empty_test)
        if empty_class_match:
            empty_class_name_in_code = empty_class_match.group(1)
            empty_test_java_file_name = f"{empty_class_name_in_code}.java"
        else:
            print("Could not find the class name in the Java code.")
            raise ValueError("Class name not found in the Java code.")

        # Save the Java code with package declaration to the file in the test package directory
        empty_test_java_file_path = os.path.join(test_package_dir, empty_test_java_file_name)
        with open(empty_test_java_file_path, 'w') as f:
            f.write(empty_test)
        print(f"Empty test Java code saved to {empty_test_java_file_path}")
        dst_java_file_path = os.path.join(dst_test_package_dir, empty_test_java_file_name)
        buggy_dst_java_file_path = os.path.join(buggy_dst_test_package_dir, empty_test_java_file_name)
        print(f"Copying {empty_test_java_file_path} to {dst_java_file_path} and {buggy_dst_java_file_path}")
        shutil.copyfile(empty_test_java_file_path, dst_java_file_path)
        shutil.copyfile(empty_test_java_file_path, buggy_dst_java_file_path)
        new_res = compile_and_test(
            f"{project_name}_{bug_id}",
            empty_test_sig,
            empty_test,
            method_name,
            "function_cov_dir",
            "second",
            tgt_model,
            strategy,
            ablation,
            format,
        )
    else:
        new_imports = new_res["passed_imports"]

        failed_classes = extract_failed_class_names(new_res["failed_imports"])
        new_fields = []
        # if fields in extractions["fields"] contains failed_classes, then do not add them to the new_fields
        for field in extractions["fields"]:
            contains_failed_class = False
            for failed_class in failed_classes:
                if failed_class in field:
                    contains_failed_class = True
                    break
            if not contains_failed_class:
                new_fields.append(field)
        new_classes = extractions["classes"]

        new_classes_methods = []
        for new_class in new_classes:
            new_class_methods = []
            for method in parse_methods_from_class_node(new_class):
                new_classes_methods.append(method["method_text"])
        new_methods = [
            method["method_text"] for method in new_res["passed_methods"] if method["method_text"] not in new_classes_methods
        ]

        second_round_test_content, second_round_test_class_sig = assemble_recursive_test_classes(
            package_name=package_name,
            imports=new_imports,
            methods=new_methods,
            fields=new_fields,
            classes=new_classes,
            class_sig=class_name
        )

        second_round_class_match = re.search(r'public\s+(?:class|interface)\s+(\w+)', second_round_test_content)
        if second_round_class_match:
            second_round_class_name_in_code = second_round_class_match.group(1)
            second_round_test_java_file_name = f"{second_round_class_name_in_code}.java"
        else:
            print("Could not find the class name in the Java code.")
            raise ValueError("Class name not found in the Java code.")

        # Save the Java code with package declaration to the file in the test package directory
        second_round_test_java_file_path = os.path.join(test_package_dir, second_round_test_java_file_name)
        with open(second_round_test_java_file_path, 'w') as f:
            f.write(second_round_test_content)
        print(f"Second round test Java code saved to {second_round_test_java_file_path}\n")

        dst_java_file_path = os.path.join(dst_test_package_dir, second_round_test_java_file_name)
        buggy_dst_java_file_path = os.path.join(buggy_dst_test_package_dir, second_round_test_java_file_name)
        print(f"Copying {second_round_test_java_file_path} to {dst_java_file_path} and {buggy_dst_java_file_path}")
        shutil.copyfile(second_round_test_java_file_path, dst_java_file_path)
        shutil.copyfile(second_round_test_java_file_path, buggy_dst_java_file_path)
        new_res = compile_and_test(
            f"{project_name}_{bug_id}",
            second_round_test_class_sig,
            second_round_test_content,
            method_name,
            "function_cov_dir",
            "second",
            tgt_model,
            strategy,
            ablation,
            format,
        )
    num_round += 1
    print(f"Round {num_round} test compilation result: {new_res['compiled']}, {new_res['passed_uts_num']} tests passed.")
    if num_round > 0:
        break


In [ ]:
new_res['compiled']

In [ ]:
new_res['passed_uts_num']

In [ ]:
new_res['coverage_results']['buggy_passed']

In [ ]:
failed_for_fixed = set()
test_command = [
    'defects4j', 'test',
    '-w', project_dir
]
print(f"Running tests from {hashed_package_name}: {' '.join(test_command)}")
test_run_log_file_path = os.path.join(test_package_dir, "test_run_log")
try:
    # Run tests and capture output
    result = subprocess.run(
        test_command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=True
    )
    # Print the captured output
    print("Defects4J output:\n" + result.stdout)
    print(f"Total number of tests generated: {num_tests}")
    # Parse the number of failing tests from the output
    failing_test_match = re.search(r"Failing tests:\s*(\d+)", result.stdout)
    if failing_test_match:
        failing_tests_count = int(failing_test_match.group(1))
        for line in result.stdout.split("\n"):
            if line.strip().startswith('-') and class_name_in_code in line:
                failed_for_fixed.add(line.strip()[1:].strip())
        print(f"Number of failing tests in LLMGenTest: {len(failed_for_fixed)}")
        print(f"Fixed LLMGenTest failing in fixed\n  - {'\n  - '.join(failed_for_fixed)}")
    else:
        print("No 'Failing tests:' line found in the output. Possibly no failures.")

    # Check return code to determine if tests passed or failed
    if result.returncode == 0 and (failing_test_match is None or failing_tests_count == 0):
        # Tests passed
        print("Tests passed successfully.")
    else:
        # Tests failed
        print(f"Tests failed with return code: {result.returncode}")

    # Save the log to a file
    with open(test_run_log_file_path, 'w') as f:
        # write total number of tests
        f.write(f"Total number of tests generated: {num_tests}\n")
        # write a separator
        f.write("-" * 80 + "\n")
        # write the output
        f.write(result.stdout)
        # write a separator
        f.write("-" * 80 + "\n")
        # write return code
        f.write(f"Return code: {result.returncode}\n")

except subprocess.CalledProcessError as e:
    # If there's an error running the command itself (e.g., command not found)
    print(f"Error running defects4j test command: {e}")
    # Save the log to a file
    with open(test_run_log_file_path, 'w') as f:
        f.write(e.stdout)
        # add a separator
        f.write("-" * 80 + "\n")
        # write return code
        f.write(f"Return code: {e.returncode}\n")
    print(f"error message: \n{e.stdout}")
    print(f"error return code: {e.returncode}")

finally:
    # Clean up: remove the Java file from the test source directory
    if os.path.exists(dst_java_file_path):
        print(f"Removing test Java file after testing: {dst_java_file_path}")
        os.remove(dst_java_file_path)

        # Optionally, remove the package directory if it is empty
        try:
            os.rmdir(dst_test_package_dir)
            print(f"Removed empty test package directory: {dst_test_package_dir}")
        except OSError:
            # Directory not empty or another error occurred
            pass


In [ ]:
parser = Parser()
parser.set_language(JAVA_LANGUAGE)

In [ ]:
def parse_classes_from_code(java_code: str) -> List[str]:
    """
    Extract the contents of all classes in the given Java code.

    :param java_code: A string containing Java source code.
    :return: A list of class texts (including decorators/annotations).
    """

    # 1. Parse the code with Tree-sitter
    tree = parser.parse(java_code.encode("utf-8"))
    root_node = tree.root_node

    # 2. Query for all class_declaration nodes
    #    (Adjust if your grammar uses a different node name for classes.)
    class_query = JAVA_LANGUAGE.query(
        """
        (class_declaration) @cls
        """
    )
    captures = class_query.captures(root_node)

    # 3. Helper function: extract the name of the class from a class_declaration
    def get_class_name_from_node(node) -> str:
        """
        Traverses the children of 'node' (which should be a class_declaration
        node). Returns the text of the 'identifier' child, which is the class name.
        """
        for child in node.children:
            if child.type == "identifier":
                return child.text.decode("utf-8")
        return ""

    # 4. Collect the text of each class
    class_texts = {}
    for (node, capture_name) in captures:
        if capture_name == "cls":
            class_name = get_class_name_from_node(node)
            # Extract the entire substring from start_byte to end_byte
            start_byte = node.start_byte
            end_byte = node.end_byte
            class_text = java_code[start_byte:end_byte]
            class_texts[class_name] = class_text

    return class_texts

def parse_methods_from_class_node_no_return_type_needed(class_str: str, need_prefix=False):
    """
    Analyze methods defined in the class_str, *without assuming* a declared return type.

    :param class_str: Java code string (optionally missing a class wrapper).
    :param need_prefix: Whether to wrap code in "public class TmpClass{\n...}\n".
    :return: list of collected methods. Each element is a dict like:
        {
            "method_name": method_name,
            "method_modifiers": method_modifiers,
            "method_return_type": method_return_type_or_empty,
            "method_body": method_body,
            "method_text": method_text,
            "method_start_line": method_start_line,
            "method_end_line": method_end_line,
            "docstring": docstring_text_or_empty
        }
    """

    parser = Parser()
    parser.set_language(JAVA_LANGUAGE)

    # Make a safe copy of the input string
    tmp_class_str = pickle.loads(pickle.dumps(class_str))

    # Optionally wrap the snippet so it's seen as a valid compilation unit
    if need_prefix:
        tmp_class_str = "public class TmpClass {\n" + tmp_class_str + "\n}"

    # Parse the code
    tree = parser.parse(tmp_class_str.encode("utf-8"))
    root_node = tree.root_node
    rets = []

    # 1. Capture all method_declaration nodes
    method_query = JAVA_LANGUAGE.query(
        """
        (method_declaration) @method_decl
        """
    )
    methods = method_query.captures(root_node)

    # 2. Make the return type optional in our attribute query:
    #    - (modifiers)? @modifier
    #    - (type:(_))? @ret_type  <-- optionally capture return type if present
    #    - name:(identifier) @name
    #    - body:(block) @body
    #
    # We'll have either 3 or 4 captures, depending on whether the type is present.
    method_attr_query = JAVA_LANGUAGE.query(
        """
        (method_declaration [
            (modifiers)? @modifier
            name:(identifier) @name
            body:(block) @body
        ])
        """
    )

    # 3. Capture all comments (line_comment OR block_comment)
    comment_query = JAVA_LANGUAGE.query(
        """
        (line_comment) @lc
        (block_comment) @bc
        """
    )
    comment_nodes = comment_query.captures(root_node)

    # Store comments in a list of dicts
    comment_list = []
    for comment_node, capture_name in comment_nodes:
        c_text = comment_node.text.decode("utf-8")
        c_start_line = comment_node.start_point[0]
        c_end_line = comment_node.end_point[0]
        comment_list.append({
            "text": c_text,
            "start_line": c_start_line,
            "end_line": c_end_line
        })

    unique_methods = set()

    # 4. Build method dicts
    for method_node, _ in methods:
        attrs = method_attr_query.captures(method_node)

        # Possible captures:
        #    - 3 captures if no return type: [ (modifier)?, (name), (body) ]
        #    - 4 captures if there's a return type: [ (modifier)?, (ret_type), (name), (body) ]
        # We'll parse them by label rather than index.
        captures_by_name = {}
        for (node_obj, node_label) in attrs:
            captures_by_name[node_label] = node_obj

        # Extract them if they exist
        modifier_node = captures_by_name.get("modifier", None)
        ret_type_node = captures_by_name.get("ret_type", None)
        name_node = captures_by_name.get("name", None)
        body_node = captures_by_name.get("body", None)

        # Must have a name and body at least
        if not name_node or not body_node:
            continue

        method_modifiers = modifier_node.text.decode("utf-8") if modifier_node else ""
        method_return_type = ret_type_node.text.decode("utf-8") if ret_type_node else ""

        method_name = name_node.text.decode("utf-8")
        method_body = body_node.text.decode("utf-8")

        method_text = method_node.text.decode("utf-8")
        method_start_line = method_node.start_point[0]
        method_end_line = method_node.end_point[0]

        # Find docstring
        docstring = ""
        closest_end_line = -1
        for c in comment_list:
            if c["end_line"] < method_start_line and c["end_line"] >= closest_end_line:
                closest_end_line = c["end_line"]
                docstring = c["text"].strip()

        # Deduplicate by method_body
        if method_body not in unique_methods and method_body.strip():
            unique_methods.add(method_body)
            rets.append({
                "method_name": method_name,
                "method_modifiers": method_modifiers,
                "method_return_type": method_return_type,  # Might be "" if no type
                "method_body": method_body,
                "method_text": method_text,
                "method_start_line": method_start_line,
                "method_end_line": method_end_line,
                "docstring": docstring,
            })

    return rets

In [ ]:
def parse_methods_from_class_node(class_str: str, need_prefix=False):
    """
    Analyze methods defined in the class_str.

    :param class_str: Java code string (optionally missing a class wrapper).
    :param need_prefix: Whether to wrap code in "public class TmpClass{\n...}\n".
    :return: list of collected methods. Each element is a dict like:
        {
            "method_name": method_name,
            "method_modifiers": method_modifiers,
            "method_return_type": method_return_type,
            "method_body": method_body,
            "method_text": method_text,
            "method_start_line": method_start_line,
            "method_end_line": method_end_line,
            "docstring": docstring_text_or_empty
        }
    """

    # Create a copy of the string so we don't mutate the original
    tmp_class_str = pickle.loads(pickle.dumps(class_str))

    # Optionally wrap in a dummy class so Tree-sitter sees a valid compilation unit
    if need_prefix:
        tmp_class_str = "public class TmpClass {\n" + tmp_class_str + "\n}"

    # Parse the code into a syntax tree
    tree = parser.parse(tmp_class_str.encode("utf-8"))
    rets = []

    # ------------------------------------------------------------------------
    # 1. CAPTURE METHOD DECLARATIONS (Just to find all 'method_declaration' nodes)
    # ------------------------------------------------------------------------
    method_query = JAVA_LANGUAGE.query(
        """
        (method_declaration) @method_decl
        """
    )
    methods = method_query.captures(tree.root_node)

    # ------------------------------------------------------------------------
    # 2. CAPTURE METHOD ATTRIBUTES (modifiers, return type, name, body)
    #    NOTE: In some versions of tree-sitter-java, return type is "_type" not "type".
    #          Adjust accordingly if you see a NameError for 'type' node.
    # ------------------------------------------------------------------------
    method_attr_query = JAVA_LANGUAGE.query(
        """
        (method_declaration [
            (modifiers) @modifier
            type:(_) @ret_type
            name:(identifier) @name
            body:(block) @body
            ])
        """
    )

    # ------------------------------------------------------------------------
    # 3. CAPTURE ALL COMMENTS (line_comment OR block_comment)
    # ------------------------------------------------------------------------
    comment_query = JAVA_LANGUAGE.query(
        """
        (line_comment) @lc
        (block_comment) @bc
        """
    )
    comment_nodes = comment_query.captures(tree.root_node)

    # Convert comment nodes into a convenient list
    # Each comment is a dict with "text", "start_line", "end_line"
    comment_list = []
    for comment_node, capture_name in comment_nodes:
        c_text = comment_node.text.decode("utf-8")
        c_start_line = comment_node.start_point[0]
        c_end_line = comment_node.end_point[0]
        comment_list.append({
            "text": c_text,
            "start_line": c_start_line,
            "end_line": c_end_line
        })

    # ------------------------------------------------------------------------
    # 4. BUILD METHOD STRUCTS
    # ------------------------------------------------------------------------
    unique_methods = set()

    for method_node, _ in methods:
        # Gather attribute captures for this particular method node
        attrs = method_attr_query.captures(method_node)

        # We expect 4 captures: (modifier, ret_type, name, body)
        # If we have a mismatch, skip this node
        if len(attrs) % 4 != 0:
            continue

        num_iter = len(attrs) // 4
        for i in range(num_iter):
            # Just a sanity check that the second capture is "ret_type"
            assert attrs[i * 4 + 1][1] == "ret_type"

            method_text = method_node.text.decode("utf-8")
            method_modifiers = attrs[i * 4][0].text.decode("utf-8")
            method_return_type = attrs[i * 4 + 1][0].text.decode("utf-8")
            method_name = attrs[i * 4 + 2][0].text.decode("utf-8")
            method_body = attrs[i * 4 + 3][0].text.decode("utf-8")

            method_start_line = method_node.start_point[0]
            method_end_line = method_node.end_point[0]

            # ----------------------------------------------------------------
            # 5. DETECT A "DOCSTRING" (closest preceding comment)
            # ----------------------------------------------------------------
            docstring = ""
            closest_end_line = -1
            for c in comment_list:
                # We want the comment that appears strictly before the method_start_line
                # but is closest. So we track the largest c["end_line"] that is < method_start_line
                if c["end_line"] < method_start_line and c["end_line"] >= closest_end_line:
                    closest_end_line = c["end_line"]
                    docstring = c["text"].strip()

            # Optional dedup logic by method_body
            if method_body not in unique_methods and method_body.strip() != "":
                unique_methods.add(method_body)
                rets.append({
                    "method_name": method_name,
                    "method_modifiers": method_modifiers,
                    "method_return_type": method_return_type,
                    "method_body": method_body,
                    "method_text": method_text,
                    "method_start_line": method_start_line,
                    "method_end_line": method_end_line,
                    "docstring": docstring,
                })

    # Finally, return the collected list of method dictionaries
    return rets

In [ ]:
def check_invocation(class_codes, test_name):
    invocated_methods = []
    for class_code_key in class_codes:
        non_constructor_methods = parse_methods_from_class_node(class_codes[class_code_key])
        non_constructor_method_no_return_type = parse_methods_from_class_node_no_return_type_needed(class_codes[class_code_key])
        all_methods_in_class = set([method['method_text'] for method in non_constructor_methods] + [method['method_text'] for method in non_constructor_method_no_return_type])
        for method_text in all_methods_in_class:
            if test_name in method_text:
                for invocation in parse_method_invocation(method_text):
                    invocated_methods.append(invocation['invoked_method_name'])
    return invocated_methods

In [ ]:
failed_for_buggy = set()
test_command = [
    'defects4j', 'test',
    '-w', buggy_project_dir
]
print(f"Running tests from {hashed_package_name}: {' '.join(test_command)}")
test_run_log_file_path = os.path.join(buggy_dst_test_package_dir, "test_run_log")
try:
    # Run tests and capture output
    result = subprocess.run(
        test_command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=True
    )

    # Print the captured output
    print("Defects4J output:\n" + result.stdout)
    print(f"Total number of tests generated: {num_tests}")
    # Parse the number of failing tests from the output
    failing_test_match = re.search(r"Failing tests:\s*(\d+)", result.stdout)
    if failing_test_match:
        failing_tests_count = int(failing_test_match.group(1))
        for line in result.stdout.split("\n"):
            if line.strip().startswith('-') and class_name_in_code in line:
                failed_for_buggy.add(line.strip()[1:].strip())
        print(f"Number of failing tests in LLMGenTest: {len(failed_for_buggy)}")
        print(f"Fixed LLMGenTest failing in buggy\n  - {'\n  - '.join(failed_for_buggy)}")
    else:
        print("No 'Failing tests:' line found in the output. Possibly no failures.")

    # Check return code to determine if tests passed or failed
    if result.returncode == 0 and (failing_test_match is None or failing_tests_count == 0):
        # Tests passed
        print("Tests passed successfully.")
    else:
        # Tests failed
        print(f"Tests failed with return code: {result.returncode}")

    # Save the log to a file
    with open(test_run_log_file_path, 'w') as f:
        # write total number of tests
        f.write(f"Total number of tests generated: {num_tests}\n")
        # write a separator
        f.write("-" * 80 + "\n")
        # write the output
        f.write(result.stdout)
        # write a separator
        f.write("-" * 80 + "\n")
        # write return code
        f.write(f"Return code: {result.returncode}\n")

except subprocess.CalledProcessError as e:
    # If there's an error running the command itself (e.g., command not found)
    print(f"Error running defects4j test command: {e}")
    # Save the log to a file
    with open(test_run_log_file_path, 'w') as f:
        f.write(e.stdout)
        # add a separator
        f.write("-" * 80 + "\n")
        # write return code
        f.write(f"Return code: {e.returncode}\n")
    print(f"error message: \n{e.stdout}")
    print(f"error return code: {e.returncode}")

finally:
    # Clean up: remove the Java file from the test source directory
    if os.path.exists(buggy_dst_java_file_path):
        print(f"Removing test Java file after testing: {buggy_dst_java_file_path}")
        os.remove(buggy_dst_java_file_path)

        # Optionally, remove the package directory if it is empty
        try:
            os.rmdir(buggy_dst_test_package_dir)
            print(f"Removed empty test package directory: {buggy_dst_test_package_dir}")
        except OSError:
            # Directory not empty or another error occurred
            pass


In [ ]:
failed_for_buggy.issubset(failed_for_fixed)

In [ ]:
test_data = data # Limit the number of items for testing

In [ ]:
# with open('/path/to/repo/data/rq1/results_test/chatgpt/fixed_test_results.json', 'r') as f:
#     last_record = json.load(f)
# failed_pre_compile = last_record['failed_pre_compile']

In [ ]:
# failed_pre_compile

In [ ]:
# test_data = [data[417]]

In [ ]:
# len(test_data)

In [ ]:
# Initialize counters
total_items = len(test_data)
total_processed = 0
partially_compiled_number = 0
non_compiled_number = 0
fully_compiled_number = 0
no_test_generated_number = 0
execute_reported_bug = 0
execute_reported_misguide = 0
execute_buggy_failed = 0
failed_pre_compile_number = 0
compiled_but_run_failed_number = 0
buggy_non_compiled_number = 0
bug_detected_number = 0
fixed_failed_but_buggy_pass_number = 0
test_generated = 0
test_compiled = 0
test_failed = 0
total_line_covered = 0
total_line_missed = 0
total_branch_covered = 0
total_branch_missed = 0
test_passed_on_buggy_but_failed_on_fixed = 0
test_passed_on_fixed_but_failed_on_buggy = 0
test_passed_on_both = 0
test_failed_on_both = 0
reported_test_passed_on_buggy_but_failed_on_fixed = 0
reported_test_passed_on_fixed_but_failed_on_buggy = 0
reported_test_passed_on_both = 0
reported_test_failed_on_both = 0
reported_passed_on_fixed = 0
reported_passed_on_buggy = 0
reported_passed_to_compiled_ratio = 0

In [ ]:
fully_compiled = []
partially_compiled = []
non_compiled = []
failed_pre_compile=[]
no_test_generated=[]
partially_passed = []
compiled_but_run_failed = []
buggy_non_compiled = []
bug_detected = []
fixed_failed_but_buggy_pass = []
cannot_obtain_coverage = []
all_line_coverage = []
all_branch_coverage = []
defects_detected = set()
all_defects = set()
defect_with_compilable_test = set()
focal_method_invoked_defects = set()
reported_defects_detected = set()
reported_defects_project = set()

In [ ]:
def setup_logging(log_dir):
    """Set up logging configuration for both main and separator loggers without console output."""
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)

    # Set up the log file
    log_files = glob.glob(os.path.join(log_dir, f'{tgt_model}_*_{input_version}_gen_test_non_private_or_default_{file_post_fix}_log_*.log'))
    if log_files:
        # Sort log files by their numeric suffix
        log_files.sort(key=lambda f: int(os.path.basename(f).split('_')[-1].split('.')[0]))
        last_log_file = log_files[-1]
        last_log_number = int(os.path.basename(last_log_file).split('_')[-1].split('.')[0])
        log_file = os.path.join(log_dir, f"{tgt_model}_gen_test_evaluate_{input_version}_gen_test_non_private_or_default_{file_post_fix}_log_{last_log_number + 1}.log")
    else:
        log_file = os.path.join(log_dir, f'{tgt_model}_gen_test_evaluate_{input_version}_gen_test_non_private_or_default_{file_post_fix}_log_1.log')
    print(f"Logging to file: {log_file}")
    # Remove any existing handlers on the root logger
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    # Setup main logger with file handler only (no console output) with levels
    logging.basicConfig(
        level=logging.INFO,
        format='%(levelname)s - %(message)s',
        filename=log_file,
        filemode='a',  # append mode
    )
    main_logger = logging.getLogger(__name__)

    # Setup separator logger to write to the same file but without level information
    separator_logger = logging.getLogger('separator_logger')
    # Remove any existing handlers from separator_logger
    for handler in separator_logger.handlers[:]:
        separator_logger.removeHandler(handler)

    # Add a file handler for the separator logger with a simple formatter that doesn't show levels
    separator_file_handler = logging.FileHandler(log_file)
    separator_file_handler.setLevel(logging.INFO)
    separator_file_handler.setFormatter(logging.Formatter('%(message)s'))
    separator_logger.addHandler(separator_file_handler)

    separator_logger.setLevel(logging.INFO)
    separator_logger.propagate = False

    return main_logger, separator_logger

logger, separator_logger = setup_logging('/path/to/repo/data/rq1/logs')

In [ ]:
# Process each item in the data, add progress bar
pbar = tqdm(test_data, desc="Processing items", unit="item", total=total_items)
for item in pbar:
    try:
        method_id = item['id']
        pbar.set_description(f"Processing {method_id}")
        # Parse project info
        project_name, bug_id = parse_project_info(method_id)
        method_signature = item.get('method_signature', '')

        logger.info(f"Processing Project: {project_name}_{bug_id}, tests generated from version: {input_version}, method signature: {method_signature}")

        # Extract the generated test from the item
        generated_test = item['completion']

        # add to all_defects
        all_defects.add(f"{project_name}_{bug_id}")

        # Set up project directory
        project_dir_name = f"{project_name}_{bug_id}/{fixed_version}"
        buggy_project_dir_name = f"{project_name}_{bug_id}/{buggy_version}"
        project_dir = os.path.join(base_checkout_dir, project_dir_name)
        buggy_project_dir = os.path.join(base_checkout_dir, buggy_project_dir_name)

        # Ensure the project directory exists
        if not os.path.exists(project_dir):
            logger.warning(f"Project directory of {project_name}_{bug_id} not found: {project_dir}. Skipping.")
            continue

        # Organize generated files in a structured way
        # Create a directory to store generated tests and related files
        generated_tests_dir = os.path.join(project_dir, 'generated_tests')
        if not os.path.exists(generated_tests_dir):
            os.makedirs(generated_tests_dir)

        # Revert to using a hashed package directory in `generated_tests`
        unique_hash = hashlib.md5(method_signature.encode('utf-8')).hexdigest()[:8]
        hashed_package_name = f"testpackage_{unique_hash}"

        # Create the directory for the test under `generated_tests` using the hashed name
        test_package_dir = os.path.join(generated_tests_dir, hashed_package_name)
        if not os.path.exists(test_package_dir):
            os.makedirs(test_package_dir)

        # Save the generated test to a Markdown file in the test package directory
        test_md_file_name = f"{method_id}_test.md"
        test_md_file_path = os.path.join(test_package_dir, test_md_file_name)
        with open(test_md_file_path, 'w') as f:
            f.write(generated_test)

        logger.info(f"Generated unit test saved to {test_md_file_path}")

        # Extract Java code from the Markdown
        code_blocks = re.findall(r'```java(.*?)```', generated_test, re.DOTALL)
        if not code_blocks:
            # Try without specifying the language
            code_blocks = re.findall(r'```(.*?)```', generated_test, re.DOTALL)

        if code_blocks:
            java_code, test_sig, extractions, (
                package_name,
                package_dir,
                class_name,
                clazz_dir,
                method_name,
                parameter_tuple,
            ) = post_process_generation(item)

            # # Count the number of test methods in the generated test
            # test_method_pattern = r'@Test\s*public\s+void\s+\w+\s*\('
            # test_methods = re.findall(test_method_pattern, java_code)
            # num_tests = len(test_methods)

            # Extract the class name from the Java code
            class_match = re.search(r'public\s+(?:class|interface)\s+(\w+)', java_code)
            if class_match:
                class_name_in_code = class_match.group(1)
                test_java_file_name = f"{class_name_in_code}.java"
            else:
                logger.warning("Could not find the class name in the Java code.")
                no_test_generated_number += 1
                no_test_generated.append((f"{project_name}_{bug_id}", method_signature))
                continue  # Skip to next item

            # Save the Java code with package declaration to the file in the test package directory
            test_java_file_path = os.path.join(test_package_dir, test_java_file_name)
            with open(test_java_file_path, 'w') as f:
                f.write(java_code)

            logger.info(f"Unit test Java code saved to {test_java_file_path}")
        else:
            logger.warning("No code block found in the generated test.")
            no_test_generated_number += 1
            no_test_generated.append((f"{project_name}_{bug_id}", method_signature))
            continue  # Skip to next item

        # Use 'defects4j export' to get the test source directory
        export_command = [
            'defects4j', 'export',
            '-p', 'dir.src.tests',
            '-w', project_dir
        ]
        try:
            result = subprocess.run(export_command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
            test_src_rel_path = result.stdout.strip()
        except subprocess.CalledProcessError as e:
            logger.error(f"Failed to get test source directory: {e}")
            continue  # Skip to next item

        export_command = [
            'defects4j', 'export',
            '-p', 'dir.src.tests',
            '-w', buggy_project_dir
        ]
        try:
            result = subprocess.run(export_command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
            buggy_test_src_rel_path = result.stdout.strip()
        except subprocess.CalledProcessError as e:
            logger.error(f"Failed to get buggy test source directory: {e}")
            continue

        # Construct the absolute path to the test source directory
        test_src_dir = os.path.join(project_dir, test_src_rel_path)
        buggy_test_src_dir = os.path.join(buggy_project_dir, buggy_test_src_rel_path)

        # Ensure the test source directory exists
        if not os.path.exists(test_src_dir):
            logger.warning(f"Test source directory not found: {test_src_dir}")
            continue  # Skip to next item

        # Ensure the buggy test source directory exists
        if not os.path.exists(buggy_test_src_dir):
            logger.warning(f"Buggy test source directory not found: {buggy_test_src_dir}")
            continue

        # Construct the destination directory in the test source directory
        dst_test_package_dir = os.path.join(test_src_dir, package_dir)
        if not os.path.exists(dst_test_package_dir):
            os.makedirs(dst_test_package_dir)

        # Construct the destination directory in the buggy test source directory
        buggy_dst_test_package_dir = os.path.join(buggy_test_src_dir, package_dir)
        if not os.path.exists(buggy_dst_test_package_dir):
            os.makedirs(buggy_dst_test_package_dir)

        # Construct the destination Java file path
        dst_java_file_path = os.path.join(dst_test_package_dir, test_java_file_name)

        # Construct the buggy destination Java file path
        buggy_dst_java_file_path = os.path.join(buggy_dst_test_package_dir, test_java_file_name)

        # Check if the test Java file already exists in the destination directory, and if so, remove it
        if os.path.exists(dst_java_file_path):
            logger.info(f"Removing existing test Java file: {dst_java_file_path}")
            os.remove(dst_java_file_path)

        # Check if the buggy test Java file already exists in the destination directory, and if so, remove it
        if os.path.exists(buggy_dst_java_file_path):
            logger.info(f"Removing existing test Java file: {buggy_dst_java_file_path}")
            os.remove(buggy_dst_java_file_path)

        # Add dependencies to the project
        add_dependencies(d4j_proj_base, f"{project_name}_{bug_id}")

        # create log files path
        pre_test_compile_log_file_path = os.path.join(test_package_dir, "pre_test_compile_log")
        buggy_pre_test_compile_log_file_path = os.path.join(test_package_dir, "buggy_pre_test_compile_log")
        test_compile_log_file_path = os.path.join(test_package_dir, "test_compile_log")
        test_run_log_file_path = os.path.join(test_package_dir, "test_run_log")
        buggy_test_run_log_file_path = os.path.join(test_package_dir, "buggy_test_run_log")
        test_coverage_log_file_path = os.path.join(test_package_dir, "test_coverage_log")
        test_coverage_json_file_path = os.path.join(test_package_dir, "test_coverage.json")

        # Do a test compile before we put any test in using Defects4J
        compile_command = [
            'defects4j', 'compile',
            '-w', project_dir
        ]

        separator_logger.info('-' * 80)
        logger.info(f"Pre-test compilation of project: {' '.join(compile_command)}")
        pbar.set_description(f"Pre-test compiling {method_id}")
        try:
            result = subprocess.run(
                compile_command,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                check=True
            )
            with open(pre_test_compile_log_file_path, 'w') as f:
                f.write(result.stdout)
            if result.returncode == 0:
                logger.info("Project pre-test compilation success.")
            else:
                logger.error("Project pre-test compilation failed.")
        except subprocess.CalledProcessError as e:
            logger.error("Pre-test compilation failed")
            with open(pre_test_compile_log_file_path, 'w') as f:
                f.write(e.stdout)
                f.write("-" * 80 + "\n")
                f.write(f"Return code: {e.returncode}\n")

            with open(test_compile_log_file_path, 'w') as f:
                f.write("Pre-test compilation failed\n")
                f.write("-" * 80 + "\n")
                f.write(e.stdout)
                f.write("-" * 80 + "\n")
                f.write(f"Return code: {e.returncode}\n")

            with open(test_run_log_file_path, 'w') as f:
                f.write("Pre-test compilation failed\n")
                f.write("-" * 80 + "\n")
                f.write(e.stdout)
                f.write("-" * 80 + "\n")
                f.write(f"Return code: {e.returncode}\n")

            failed_pre_compile_number += 1
            failed_pre_compile.append((f"{project_name}_{bug_id}", method_signature))

            continue  # Skip to next item

        # Do a test of the buggy project before we put any test in using Defects4J
        compile_command = [
            'defects4j', 'compile',
            '-w', buggy_project_dir
        ]

        separator_logger.info('-' * 80)
        logger.info(f"Pre-test compilation of buggy project: {' '.join(compile_command)}")
        pbar.set_description(f"Pre-test compiling buggy {method_id}")
        try:
            result = subprocess.run(
                compile_command,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                check=True
            )
            with open(buggy_pre_test_compile_log_file_path, 'w') as f:
                f.write(result.stdout)
            if result.returncode == 0:
                logger.info("Buggy project pre-test compilation success.")
            else:
                logger.error("Buggy project pre-test compilation failed.")
        except subprocess.CalledProcessError as e:
            logger.error("Buggy pre-test compilation failed")
            with open(buggy_pre_test_compile_log_file_path, 'w') as f:
                # add a separator
                f.write("-" * 80 + "\n")
                f.write(e.stdout)
                f.write("-" * 80 + "\n")
                f.write("Fixed pre-test compilation failed\n")
                f.write(f"Return code: {e.returncode}\n")

            buggy_non_compiled_number += 1
            buggy_non_compiled.append((f"{project_name}_{bug_id}", method_signature))

        logger.info(f"Copying {test_java_file_path} to {dst_java_file_path} and {buggy_dst_java_file_path}")
        shutil.copyfile(test_java_file_path, dst_java_file_path)
        shutil.copyfile(test_java_file_path, buggy_dst_java_file_path)

        # set the number of compilation rounds to 0
        num_round = 0

        separator_logger.info('-' * 80)
        logger.info(f"Compiling project with test from {hashed_package_name}: {' '.join(compile_command)}")

        pbar.set_description(f"Compiling tests of {method_id}")
        # Start loop to compile the project with the generated test

        res = compile_and_test(
            f"{project_name}_{bug_id}",
            test_sig,
            java_code,
            method_name,
            "class_cov_dir",
            "first",
            tgt_model,
            strategy,
            ablation,
            format,
        )

        item['compiled'] = res['compiled']
        item['compile_err_msg'] = res['err_msg']

        num_tests = res['total_uts_num']
        test_generated += num_tests

        if res.get("JacocoFailedException") is not None:
            logger.error(str(res.get("JacocoFailedException")))
            # res = {
            #     'compiled': True,
            #     'passed_uts_num': num_tests,
            #     'err_msg': str(e),
            #     'coverage_results': {
            #         'fixed_coverage': {},
            #     },
            #     'total_uts_num': num_tests
            # }

        logger.info(f"Test compilation result at round {num_round}: {res['compiled']}, {res['passed_uts_num']} tests passed.")
        with open(test_compile_log_file_path, 'w') as f:
            f.write(f"Test compilation result at round {num_round}: {res['compiled']}, {res['passed_uts_num']} tests passed.\n")
            if not res['compiled']:
                f.write(f"Error message: \n{res['err_msg']}\n")
            f.write("_" * 80 + "\n")

        new_res = copy.deepcopy(res)

        # If the test did not compile, try to generate a new test
        while not new_res["compiled"]:
            num_round += 1
            # clean up the non-compilable test
            if os.path.exists(dst_java_file_path):
                logger.warning(f"Removing test Java file after compilation failure: {dst_java_file_path}")
                os.remove(dst_java_file_path)

            # clean up the buggy non-compilable test
            if os.path.exists(buggy_dst_java_file_path):
                logger.warning(f"Removing buggy test Java file after compilation failure: {buggy_dst_java_file_path}")
                os.remove(buggy_dst_java_file_path)

            if new_res["passed_uts_num"] == 0:
                empty_test, empty_test_sig = assemble_empty_test_file(class_name)
                empty_class_match = re.search(r'public\s+(?:class|interface)\s+(\w+)', empty_test)
                if empty_class_match:
                    empty_class_name_in_code = empty_class_match.group(1)
                    empty_test_java_file_name = f"{empty_class_name_in_code}.java"
                else:
                    logger.warning("Could not find the class name in the Java code.")
                    break

                empty_test_java_file_path = os.path.join(test_package_dir, empty_test_java_file_name)
                with open(empty_test_java_file_path, 'w') as f:
                    f.write(empty_test)
                logger.info(f"Empty test Java code saved to {empty_test_java_file_path}")
                dst_java_file_path = os.path.join(dst_test_package_dir, empty_test_java_file_name)
                buggy_dst_java_file_path = os.path.join(buggy_dst_test_package_dir, empty_test_java_file_name)
                logger.info(f"Copying {empty_test_java_file_path} to {dst_java_file_path}" and {buggy_dst_java_file_path})
                shutil.copyfile(empty_test_java_file_path, dst_java_file_path)
                shutil.copyfile(empty_test_java_file_path, buggy_dst_java_file_path)

                new_res = compile_and_test(
                    f"{project_name}_{bug_id}",
                    empty_test_sig,
                    empty_test,
                    method_name,
                    "class_cov_dir",
                    "second",
                    tgt_model,
                    strategy,
                    ablation,
                    format,
                )
                if new_res.get("JacocoFailedException") is not None:
                    logger.error(str(new_res.get("JacocoFailedException")))
                    # new_res['compiled'] = True
                    # new_res['coverage_results'] = {
                    #     'fixed_coverage': {},
                    # }
                new_res["passed_uts_num"] = 0
                with open(test_compile_log_file_path, 'a') as f:
                    f.write(f"Test compilation result at round {num_round}: {new_res['compiled']}, {new_res['passed_uts_num']} tests passed.\n")
                    if not new_res['compiled']:
                        f.write(f"Error message: \n{new_res['err_msg']}\n")
                    f.write("_" * 80 + "\n")
            else:
                new_imports = new_res["passed_imports"]
                # new_methods = [
                #     method["method_text"] for method in new_res["passed_methods"]
                # ]
                failed_classes = extract_failed_class_names(new_res["failed_imports"])
                new_fields = []
                for field in extractions["fields"]:
                    contains_failed_class = False
                    for failed_class in failed_classes:
                        if failed_class in field:
                            contains_failed_class = True
                            break
                    if not contains_failed_class:
                        new_fields.append(field)
                new_classes = extractions["classes"]
                new_classes_methods = []
                for new_class in new_classes:
                    new_class_methods = []
                    for method in parse_methods_from_class_node(new_class):
                        new_classes_methods.append(method["method_text"])
                new_methods = [
                    method["method_text"] for method in new_res["passed_methods"] if method["method_text"] not in new_classes_methods
                ]

                second_round_test_content, second_round_test_class_sig = assemble_recursive_test_classes(
                    package_name=package_name,
                    imports=new_imports,
                    methods=new_methods,
                    fields=new_fields,
                    classes=new_classes,
                    class_sig=class_name
                )

                second_round_class_match = re.search(r'public\s+(?:class|interface)\s+(\w+)', second_round_test_content)
                if second_round_class_match:
                    second_round_class_name_in_code = second_round_class_match.group(1)
                    second_round_test_java_file_name = f"{second_round_class_name_in_code}.java"
                else:
                    logger.warning("Could not find the class name in the Java code.")
                    new_res["passed_uts_num"] = 0
                    break

                second_round_test_java_file_path = os.path.join(test_package_dir, second_round_test_java_file_name)
                with open(second_round_test_java_file_path, 'w') as f:
                    f.write(second_round_test_content)
                logger.info(f"Second round test Java code saved to {second_round_test_java_file_path}")
                dst_java_file_path = os.path.join(dst_test_package_dir, second_round_test_java_file_name)
                buggy_dst_java_file_path = os.path.join(buggy_dst_test_package_dir, second_round_test_java_file_name)
                logger.info(f"Copying {second_round_test_java_file_path} to {dst_java_file_path}")
                shutil.copyfile(second_round_test_java_file_path, dst_java_file_path)
                shutil.copyfile(second_round_test_java_file_path, buggy_dst_java_file_path)

                new_res = compile_and_test(
                    f"{project_name}_{bug_id}",
                    second_round_test_class_sig,
                    second_round_test_content,
                    method_name,
                    "function_cov_dir",
                    "second",
                    tgt_model,
                    strategy,
                    ablation,
                    format,
                )
                if new_res.get("JacocoFailedException") is not None:
                    logger.error(str(new_res.get("JacocoFailedException")))
                    # new_res['compiled'] = True
                    # new_res['coverage_results'] = {
                    #     'fixed_coverage': {},
                    # }
                with open(test_compile_log_file_path, 'a') as f:
                    f.write(f"Test compilation result at round {num_round}: {new_res['compiled']}, {new_res['passed_uts_num']} tests passed.\n")
                    if not new_res['compiled']:
                        f.write(f"Error message: \n{new_res['err_msg']}\n")
                    f.write("_" * 80 + "\n")

            logger.info(f"Round {num_round} test compilation result: {new_res['compiled']}, {new_res['passed_uts_num']} tests passed.")

        item['ran'] = True
        if new_res['coverage_results']:
            # add a separator
            separator_logger.info('-' * 80)
            logger.info("Reported execution results:")
            logger.info(f"buggy passed {new_res['coverage_results']['buggy_passed']}, fixed passed {new_res['coverage_results']['fixed_passed']}")

            # reported_buggy_test_error = []
            reported_buggy_test_error_dict = {}
            if new_res['coverage_results']["buggy_error_report"]!= "":
                buggy_error_report_lines = new_res['coverage_results']["buggy_error_report"].split("\n")
                logger.info("buggy error report:")
                for idx, line in enumerate(buggy_error_report_lines):
                    if line.startswith('--- '):
                        # reported_buggy_test_error.append((line, buggy_error_report_lines[idx + 1]))
                        reported_buggy_test_error_dict[line] = buggy_error_report_lines[idx + 1]
                        logger.info(line)
                        logger.info(buggy_error_report_lines[idx + 1])
            else:
                logger.info("buggy error empty")

            # reported_fixed_test_error = []
            reported_fixed_test_error_dict = {}
            if new_res['coverage_results']["fixed_error_report"]!= "":
                fixed_error_report_lines = new_res['coverage_results']["fixed_error_report"].split("\n")
                logger.info("fixed error report:")
                for idx, line in enumerate(fixed_error_report_lines):
                    if line.startswith('--- '):
                        # reported_fixed_test_error.append((line, fixed_error_report_lines[idx + 1]))
                        reported_fixed_test_error_dict[line] = fixed_error_report_lines[idx + 1]
                        logger.info(line)
                        logger.info(fixed_error_report_lines[idx + 1])
            else:
                logger.info("fixed error empty")

            # reported_buggy_test_error = set(reported_buggy_test_error)
            # reported_fixed_test_error = set(reported_fixed_test_error)
            reported_buggy_test_error = set(reported_buggy_test_error_dict.keys())
            reported_fixed_test_error = set(reported_fixed_test_error_dict.keys())

            if not new_res['coverage_results']['buggy_passed']:
                execute_buggy_failed += 1
                if reported_buggy_test_error:
                    reported_passed_on_buggy += new_res["passed_uts_num"] - len(reported_buggy_test_error)
            else:
                reported_passed_on_buggy += new_res["passed_uts_num"]

            if not new_res['coverage_results']['fixed_passed']:
                if reported_fixed_test_error:
                    reported_passed_on_fixed += new_res["passed_uts_num"] - len(reported_fixed_test_error)
            else:
                reported_passed_on_fixed += new_res["passed_uts_num"]

            if new_res['coverage_results']['buggy_passed'] and new_res['coverage_results']['fixed_passed']:
                logger.info(f"bug id: {project_name}_{bug_id}, method signature: {method_signature} passed on both")
                reported_test_passed_on_both += new_res["passed_uts_num"]

            if new_res['coverage_results']['buggy_passed'] != new_res['coverage_results']['fixed_passed']:
                if new_res['coverage_results']['buggy_passed']:
                    execute_reported_misguide += 1
                    logger.info(f"bug id: {project_name}_{bug_id}, method signature: {method_signature} misguided")
                    if reported_fixed_test_error:
                        reported_test_passed_on_buggy_but_failed_on_fixed += len(reported_fixed_test_error)
                        reported_test_passed_on_both += new_res["passed_uts_num"] - len(reported_fixed_test_error)
                    else:
                        reported_test_passed_on_buggy_but_failed_on_fixed += new_res["passed_uts_num"]
                else:
                    execute_reported_bug += 1
                    logger.info(f"bug id: {project_name}_{bug_id}, method signature: {method_signature} bug")
                    if f"{project_name}_{bug_id}" not in reported_defects_detected:
                        reported_defects_detected.add(f"{project_name}_{bug_id}")
                        logger.info(f"New defect: {project_name}_{bug_id} reported by test from method {method_signature}")
                    if project_name not in reported_defects_project:
                        reported_defects_project.add(project_name)
                        logger.info(f"New buggy project: {project_name} reported by test from method {method_signature}")
                    if reported_buggy_test_error:
                        reported_test_passed_on_fixed_but_failed_on_buggy += len(reported_buggy_test_error)
                        reported_test_passed_on_both += new_res["passed_uts_num"] - len(reported_buggy_test_error)
                    else:
                        reported_test_passed_on_fixed_but_failed_on_buggy += new_res["passed_uts_num"]

            if not new_res['coverage_results']['buggy_passed'] and not new_res['coverage_results']['fixed_passed']:
                if reported_buggy_test_error and reported_fixed_test_error:
                    if reported_buggy_test_error != reported_fixed_test_error:
                        if not reported_fixed_test_error.issubset(reported_buggy_test_error):
                            execute_reported_misguide += 1
                            reported_test_passed_on_buggy_but_failed_on_fixed += len(reported_fixed_test_error - reported_buggy_test_error)
                            logger.info(f"bug id: {project_name}_{bug_id}, method signature: {method_signature} misguided")
                        if not reported_buggy_test_error.issubset(reported_fixed_test_error):
                            execute_reported_bug += 1
                            logger.info(f"bug id: {project_name}_{bug_id}, method signature: {method_signature} bug")
                            if f"{project_name}_{bug_id}" not in reported_defects_detected:
                                reported_defects_detected.add(f"{project_name}_{bug_id}")
                                logger.info(f"New defect: {project_name}_{bug_id} reported by test from method {method_signature}")
                            if project_name not in reported_defects_project:
                                reported_defects_project.add(project_name)
                                logger.info(f"New buggy project: {project_name} reported by test from method {method_signature}")
                            reported_test_passed_on_fixed_but_failed_on_buggy += len(reported_buggy_test_error - reported_fixed_test_error)
                    else:
                        logger.info(f"bug id: {project_name}_{bug_id}, method signature: {method_signature} buggy and fixed failed the same tests")
                    reported_test_passed_on_both += new_res["passed_uts_num"] - len(reported_buggy_test_error.union(reported_fixed_test_error))
                    reported_test_failed_on_both += len(reported_buggy_test_error.intersection(reported_fixed_test_error))
                else:
                    if reported_buggy_test_error:
                        execute_reported_misguide += 1
                        logger.info(f"bug id: {project_name}_{bug_id}, method signature: {method_signature} misguided")
                        reported_test_passed_on_buggy_but_failed_on_fixed += new_res["passed_uts_num"] - len(reported_buggy_test_error)
                        reported_test_failed_on_both += len(reported_buggy_test_error)
                    elif reported_fixed_test_error:
                        execute_reported_bug += 1
                        logger.info(f"bug id: {project_name}_{bug_id}, method signature: {method_signature} bug")
                        if f"{project_name}_{bug_id}" not in reported_defects_detected:
                            reported_defects_detected.add(f"{project_name}_{bug_id}")
                            logger.info(f"New defect: {project_name}_{bug_id} reported by test from method {method_signature}")
                        if project_name not in reported_defects_project:
                            reported_defects_project.add(project_name)
                            logger.info(f"New buggy project: {project_name} reported by test from method {method_signature}")
                        reported_test_passed_on_fixed_but_failed_on_buggy += new_res["passed_uts_num"] - len(reported_fixed_test_error)
                        reported_test_failed_on_both += len(reported_fixed_test_error)
                    else:
                        reported_test_failed_on_both += new_res["passed_uts_num"]

            # add a separator
            separator_logger.info('-' * 80)

        try:
            # get the coverage results
            focal_method = item['focal_method']
            coverage_stats = calculate_coverage_stats(
                focal_method,
                new_res['coverage_results']['fixed_coverage'],
                method_name,
                package_dir,
                clazz_dir,
                parameter_tuple,
            )
            line_covered = coverage_stats['line_coverage_covered']
            line_missed = coverage_stats['line_coverage_missed']
            branch_covered = coverage_stats['branch_coverage_covered']
            branch_missed = coverage_stats['branch_coverage_missed']

            logger.info(f"Coverage results for {hashed_package_name}:")
            logger.info(f"Line covered: {line_covered}, Line missed: {line_missed}")
            logger.info(f"Branch covered: {branch_covered}, Branch missed: {branch_missed}")

            if line_covered != -1 and line_missed != -1:
                line_coverage = line_covered / (line_covered + line_missed) * 100
                total_line_covered += line_covered
                total_line_missed += line_missed
                all_line_coverage.append(line_coverage)
                logger.info(f"Line coverage: {line_coverage:.2f}%")
            else:
                line_coverage = -1
                logger.warning("Line coverage not available.")
            if branch_covered != -1 and branch_missed != -1:
                branch_cov = branch_covered / (branch_covered + branch_missed) * 100
                total_branch_covered += branch_covered
                total_branch_missed += branch_missed
                all_branch_coverage.append(branch_cov)
                logger.info(f"Branch coverage: {branch_cov:.2f}%")
            else:
                branch_cov = -1
                logger.warning("Branch coverage not available.")

            with open(test_coverage_log_file_path, 'w') as f:
                f.write(f"Line covered: {line_covered}, Line missed: {line_missed}\n")
                f.write(f"Branch covered: {branch_covered}, Branch missed: {branch_missed}\n")
                f.write(f"Line coverage: {line_coverage:.2f}%\n")
                f.write(f"Branch coverage: {branch_cov:.2f}%\n")
            logger.info(f"Coverage stats saved to {test_coverage_log_file_path}")

            with open(test_coverage_json_file_path, 'w') as f:
                json.dump(coverage_stats, f, indent=4)
            logger.info(f"Coverage results saved to {test_coverage_json_file_path}")

            separator_logger.info('-' * 80)
        except Exception as e:
            logger.exception(f"Coverage results for {hashed_package_name} not available")
            cannot_obtain_coverage.append((f"{project_name}_{bug_id}", method_signature))
            with open(test_coverage_log_file_path, 'w') as f:
                f.write("Coverage results not available\n")
                f.write(f"Error msg: {e}")
            separator_logger.info('-' * 80)

        test_compiled += new_res["passed_uts_num"]
        if new_res["passed_uts_num"] == 0:
            non_compiled_number += 1
            non_compiled.append((f"{project_name}_{bug_id}", method_signature))
        if 0 < new_res["passed_uts_num"] < res['total_uts_num']:
            partially_compiled_number += 1
            partially_compiled.append((f"{project_name}_{bug_id}", method_signature))
        if new_res["passed_uts_num"] == res['total_uts_num']:
            fully_compiled_number += 1
            fully_compiled.append((f"{project_name}_{bug_id}", method_signature))

        if new_res["passed_uts_num"] != 0:
            defect_with_compilable_test.add(f"{project_name}_{bug_id}")
            fixed_test_ran = True
            failed_for_fixed = set()
            test_command = [
                'defects4j', 'test',
                '-w', project_dir
            ]
            pbar.set_description(f"Running tests of {method_id}")
            logger.info(f"Running tests from {hashed_package_name}: {' '.join(test_command)}")
            try:
                result = subprocess.run(
                    test_command,
                    stdout=subprocess.PIPE,
                    stderr=subprocess.STDOUT,
                    text=True,
                    check=True
                )

                logger.info(result.stdout)
                logger.info(f"Total number of tests generated: {num_tests}")
                logger.info(f"Total number of tests compiled: {new_res['passed_uts_num']}")
                failing_test_match = re.search(r"Failing tests:\s*(\d+)", result.stdout)
                if failing_test_match:
                    failing_tests_count = int(failing_test_match.group(1))
                    partially_passed.append((f"{project_name}_{bug_id}", method_signature))
                    for line in result.stdout.split("\n"):
                        if line.strip().startswith('-') and class_name_in_code in line:
                            failed_for_fixed.add(line.strip()[1:].strip())
                    logger.info(f"Number of failing tests in LLMGenTest: {len(failed_for_fixed)}")
                    if len(failed_for_fixed) > 0:
                        logger.info("Fixed LLMGenTest failing in fixed\n  - " + '\n  - '.join(failed_for_fixed))
                        test_failed += len(failed_for_fixed)
                    else:
                        logger.info("Fixed LLMGenTest all passed in fixed")

                else:
                    logger.info("No 'Failing tests:' line found in the output. Possibly no failures.")

                if result.returncode == 0:
                    logger.info("Tests ran successfully.")
                else:
                    item['ran'] = False
                    item['run_err_msg'] = result.stdout
                    logger.error(f"Tests failed with return code: {result.returncode}")

                with open(test_run_log_file_path, 'w') as f:
                    f.write(f"Total number of tests generated: {num_tests}\n")
                    f.write("-" * 80 + "\n")
                    f.write(result.stdout)
                    f.write("-" * 80 + "\n")
                    f.write(f"Return code: {result.returncode}\n")

            except subprocess.CalledProcessError as e:
                fixed_test_ran = False
                logger.error(f"Error running defects4j test command: {e}")
                compiled_but_run_failed_number += 1
                compiled_but_run_failed.append((f"{project_name}_{bug_id}", method_signature))
                with open(test_run_log_file_path, 'w') as f:
                    f.write(e.stdout)
                    f.write("-" * 80 + "\n")
                    f.write(f"Return code: {e.returncode}\n")
                logger.error(f"error message: \n{e.stdout}")
                logger.error(f"error return code: {e.returncode}")
                item['ran'] = False
                item['run_err_msg'] = e.stdout

            finally:
                if os.path.exists(dst_java_file_path):
                    logger.info(f"Removing test Java file after testing: {dst_java_file_path}")
                    os.remove(dst_java_file_path)
                    try:
                        os.rmdir(dst_test_package_dir)
                        logger.info(f"Removed empty test package directory: {dst_test_package_dir}")
                    except OSError:
                        pass
            buggy_test_ran = True
            failed_for_buggy = set()
            if new_res['buggy_compiled']:
                # add a separator
                separator_logger.info('-' * 80)
                buggy_test_command = [
                    'defects4j', 'test',
                    '-w', buggy_project_dir
                ]
                pbar.set_description(f"Running buggy tests of {method_id}")
                logger.info(f"Running buggy tests from {hashed_package_name}: {' '.join(buggy_test_command)}")
                try:
                    result = subprocess.run(
                        buggy_test_command,
                        stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT,
                        text=True,
                        check=True
                    )

                    logger.info(result.stdout)
                    logger.info(f"Total number of tests generated: {num_tests}")
                    logger.info(f"Total number of tests compiled: {new_res['passed_uts_num']}")
                    failing_test_match = re.search(r"Failing tests:\s*(\d+)", result.stdout)
                    if failing_test_match:
                        failing_tests_count = int(failing_test_match.group(1))
                        for line in result.stdout.split("\n"):
                            if line.strip().startswith('-'):
                                cleaned_line = line.strip()[1:].strip()
                                if class_name_in_code in line:
                                    failed_for_buggy.add(line.strip()[1:].strip())
                                else:
                                    if len(cleaned_line.split('::')) < 2:
                                        logger.warning(f"Error line not in the expected format: {cleaned_line}")
                                        continue
                                    trig_test_class = cleaned_line.split('::')[0]
                                    trig_test_name = cleaned_line.split('::')[1]
                                    trig_test_class_dir = trig_test_class.replace('.', os.sep)
                                    full_trig_test_path = os.path.join(buggy_test_src_dir, trig_test_class_dir+".java")
                                    full_trig_test_path_bak = os.path.join(test_src_dir, trig_test_class_dir+".java.bak")
                                    try:
                                        with open(full_trig_test_path, 'r', encoding='utf-8', errors='replace') as f:
                                            test_content = f.read()
                                    except FileNotFoundError:
                                        try: # try to read the backup file
                                            with open(full_trig_test_path_bak, 'r', encoding='utf-8', errors='replace') as f:
                                                test_content = f.read()
                                        except FileNotFoundError:
                                            logger.warning(f"Test file not found: {full_trig_test_path} and {full_trig_test_path_bak}")
                                            continue
                                    if method_name in check_invocation(parse_classes_from_code(test_content), trig_test_name):
                                        focal_method_invoked_defects.add(f"{project_name}_{bug_id}")
                                        logger.info(f"Focal method {method_name} invoked in {trig_test_class}::{trig_test_name}")
                                        break

                        logger.info(f"Number of failing tests in LLMGenTest: {len(failed_for_buggy)}")
                        if len(failed_for_buggy) > 0:
                            logger.info("Fixed LLMGenTest failing in buggy\n  - " + '\n  - '.join(failed_for_buggy))
                        else:
                            logger.info("Fixed LLMGenTest all passed in buggy")
                    else:
                        logger.info("No 'Failing tests:' line found in the output. Possibly no failures.")

                    if result.returncode == 0:
                        logger.info("Tests ran successfully.")
                    else:
                        logger.error(f"Tests failed with return code: {result.returncode}")

                    with open(buggy_test_run_log_file_path, 'w') as f:
                        f.write(f"Total number of tests generated: {num_tests}\n")
                        f.write("-" * 80 + "\n")
                        f.write(result.stdout)
                        f.write("-" * 80 + "\n")
                        f.write(f"Return code: {result.returncode}\n")

                except subprocess.CalledProcessError as e:
                    buggy_test_ran = False
                    logger.error(f"Error running defects4j test command: {e}")
                    with open(buggy_test_run_log_file_path, 'w') as f:
                        f.write(e.stdout)
                        f.write("-" * 80 + "\n")
                        f.write(f"Return code: {e.returncode}\n")
                    logger.error(f"error message: \n{e.stdout}")
                    logger.error(f"error return code: {e.returncode}")

                finally:
                    if os.path.exists(buggy_dst_java_file_path):
                        logger.info(f"Removing buggy test Java file after testing: {buggy_dst_java_file_path}")
                        os.remove(buggy_dst_java_file_path)
                        try:
                            os.rmdir(buggy_dst_test_package_dir)
                            logger.info(f"Removed empty test package directory: {buggy_dst_test_package_dir}")
                        except OSError:
                            pass
            else:
                logger.info("Buggy test did not compile")
                buggy_test_ran = False
            # add a separator
            separator_logger.info('-' * 80)
            if fixed_test_ran:
                if buggy_test_ran and not failed_for_buggy.issubset(failed_for_fixed):
                    logger.info(f"bug detected")
                    bug_detected.append((f"{project_name}_{bug_id}", method_signature))
                    bug_detected_number += 1
                    if f"{project_name}_{bug_id}" not in defects_detected:
                        logger.info(f"new defect detected: {project_name}_{bug_id}")
                        defects_detected.add(f"{project_name}_{bug_id}")
                    if len(failed_for_fixed - failed_for_buggy) > 0:
                        test_passed_on_buggy_but_failed_on_fixed += len(failed_for_fixed - failed_for_buggy)
                        fixed_failed_but_buggy_pass.append((f"{project_name}_{bug_id}", method_signature))
                    if len(failed_for_buggy - failed_for_fixed) > 0:
                        test_passed_on_fixed_but_failed_on_buggy += len(failed_for_buggy - failed_for_fixed)

                elif buggy_test_ran and failed_for_buggy.issubset(failed_for_fixed):
                    logger.info(f"bug not detected")
                    if len(failed_for_fixed - failed_for_buggy) > 0:
                        test_passed_on_buggy_but_failed_on_fixed += len(failed_for_fixed - failed_for_buggy)
                        fixed_failed_but_buggy_pass.append((f"{project_name}_{bug_id}", method_signature))
                        fixed_failed_but_buggy_pass_number += 1
                    if len(failed_for_buggy - failed_for_fixed) > 0:
                        test_passed_on_fixed_but_failed_on_buggy += len(failed_for_buggy - failed_for_fixed)
                else:
                    logger.info("bug detected")
                    if f"{project_name}_{bug_id}" not in defects_detected:
                        logger.info(f"new defect detected: {project_name}_{bug_id}")
                        defects_detected.add(f"{project_name}_{bug_id}")
                    bug_detected.append((f"{project_name}_{bug_id}", method_signature))
                    bug_detected_number += 1
                    buggy_non_compiled_number += 1
                    buggy_non_compiled.append((f"{project_name}_{bug_id}", method_signature))
                    test_passed_on_fixed_but_failed_on_buggy += (new_res["passed_uts_num"] - len(failed_for_fixed))

            else:
                logger.info("Fixed test did not run")
                logger.info("bug not detected")
                if buggy_test_ran:
                    fixed_failed_but_buggy_pass.append((f"{project_name}_{bug_id}", method_signature))
                    fixed_failed_but_buggy_pass_number += 1
                    test_passed_on_buggy_but_failed_on_fixed += (new_res["passed_uts_num"] - len(failed_for_buggy))
            test_passed_on_both += (new_res["passed_uts_num"] - len(failed_for_fixed.union(failed_for_buggy)))
            test_failed_on_both += len(failed_for_fixed.intersection(failed_for_buggy))

        else:
            logger.info("Empty Test, no need to run")
            with open(test_run_log_file_path, 'w') as f:
                f.write("Empty Test, no need to run\n")

            if os.path.exists(dst_java_file_path):
                logger.info(f"Removing test Java file after testing: {dst_java_file_path}")
                os.remove(dst_java_file_path)
                try:
                    os.rmdir(dst_test_package_dir)
                    logger.info(f"Removed empty test package directory: {dst_test_package_dir}")
                except OSError:
                    pass

            if os.path.exists(buggy_dst_java_file_path):
                logger.info(f"Removing buggy test Java file after testing: {buggy_dst_java_file_path}")
                os.remove(buggy_dst_java_file_path)
                try:
                    os.rmdir(buggy_dst_test_package_dir)
                    logger.info(f"Removed empty test package directory: {buggy_dst_test_package_dir}")
                except OSError:
                    pass

    finally:
        separator_logger.info('-' * 80)
        total_processed += 1
        logger.info(f"Total processed: {total_processed}")
        logger.info(f"Total defect processed: {len(all_defects)}")
        logger.info(f"All test compiled: {fully_compiled_number}")
        logger.info(f"Part of the test compiled: {partially_compiled_number}")
        logger.info(f"None of the test compiled: {non_compiled_number}")
        logger.info(f"No test generated: {no_test_generated_number}")
        logger.info(f"Failed to compile before adding any test: {failed_pre_compile_number}")
        logger.info(f"Tests compiled but failed to run: {compiled_but_run_failed_number}")
        logger.info(f"Bug detected: {bug_detected_number}")
        logger.info(f"Defects detected: {len(defects_detected)}")
        logger.info(f"Defect have compilable test: {len(defect_with_compilable_test)}")
        logger.info(f"Defect with invocations of focal method: {len(focal_method_invoked_defects)}")
        logger.info(f"Buggy non-compiled: {buggy_non_compiled_number}")
        logger.info(f"Fixed failed but buggy pass: {fixed_failed_but_buggy_pass_number}")
        logger.info(f"test passed on buggy but failed on fixed: {test_passed_on_buggy_but_failed_on_fixed}")
        logger.info(f"test passed on fixed but failed on buggy: {test_passed_on_fixed_but_failed_on_buggy}")
        logger.info(f"test passed on both: {test_passed_on_both}")
        logger.info(f"test failed on both: {test_failed_on_both}")
        logger.info(f"Misguided: {execute_reported_misguide}")
        logger.info(f"Reported Bug: {execute_reported_bug}")
        logger.info(f"Failed test in buggy: {execute_buggy_failed}")
        logger.info(f"Reported defects detected: {len(reported_defects_detected)}")
        logger.info(f"Reported projects detected: {len(reported_defects_project)}")
        logger.info(f"Total test generated: {test_generated}")
        logger.info(f"Total test successfully compiled: {test_compiled}")
        logger.info(f"Total test compiled but failed: {test_failed}")
        logger.info(f"reported test passed on buggy but failed on fixed: {reported_test_passed_on_buggy_but_failed_on_fixed}")
        logger.info(f"reported test passed on fixed but failed on buggy: {reported_test_passed_on_fixed_but_failed_on_buggy}")
        logger.info(f"reported test passed on both: {reported_test_passed_on_both}")
        logger.info(f"reported test failed on both: {reported_test_failed_on_both}")
        logger.info(f"reported test passed on fixed: {reported_passed_on_fixed}")
        logger.info(f"reported test passed on buggy: {reported_passed_on_buggy}")
        logger.info(f"reported test passed to compilable rate: {reported_passed_on_fixed/test_compiled}")
        logger.info(f"reported test passed to generated rate: {reported_passed_on_fixed/test_generated}")

        if test_generated > 0:
            logger.info(f"Compilable rate: {test_compiled/test_generated}")
            logger.info(f"Runnable to generated rate: {(test_compiled-test_failed)/test_generated}")
        else:
            logger.info("Compilable rate not available.")
            logger.info("Runnable to generated rate not available.")
        if test_compiled > 0:
            logger.info(f"Runnable to compilable rate: {(test_compiled-test_failed)/test_compiled}")
        else:
            logger.info("Runnable to compilable rate not available.")
        if total_line_covered + total_line_missed != 0:
            logger.info(f"Total line coverage: {total_line_covered/(total_line_covered+total_line_missed)}")
        else:
            logger.info("Total line coverage not available.")
        if total_branch_covered + total_branch_missed != 0:
            logger.info(f"Total branch coverage: {total_branch_covered/(total_branch_covered+total_branch_missed)}")
        else:
            logger.info("Total branch coverage not available.")
        if len(all_line_coverage) != 0:
            logger.info(f"Average line coverage: {np.mean(all_line_coverage)}")
        else:
            logger.info("Average line coverage not available.")
        if len(all_branch_coverage) != 0:
            logger.info(f"Average branch coverage: {np.mean(all_branch_coverage)}")
        else:
            logger.info("Average branch coverage not available.")

        separator_logger.info('=' * 80)


In [ ]:
# store updated data with err message
jsonl_file_with_err_msg_path = f'/path/to/repo/data/prompts/rq1/{tgt_model}_{format}_{strategy}_{ablation}_{input_version}_{file_post_fix}_err_msg_with_completion.jsonl'

In [ ]:
with open(jsonl_file_with_err_msg_path, 'w') as f:
    for item in test_data:
        f.write(json.dumps(item) + '\n')

In [ ]:
print(f"Total processed: {total_processed}")
print(f"Total defect processed: {len(all_defects)}")
print(f"All test compiled: {fully_compiled_number}")
print(f"Part of the test compiled: {partially_compiled_number}")
print(f"None of the test compiled: {non_compiled_number}")
print(f"No test generated: {no_test_generated_number}")
print(f"Failed to compile before adding any test: {failed_pre_compile_number}")
print(f"Tests compiled but failed to run: {compiled_but_run_failed_number}")
print(f"Bug detected: {bug_detected_number}")
print(f"Defects detected: {len(defects_detected)}")
print(f"Defect have compilable test: {len(defect_with_compilable_test)}")
print(f"Defect with invocation of focal method: {len(focal_method_invoked_defects)}")
print(f"Buggy non-compiled: {buggy_non_compiled_number}")
print(f"Fixed failed but buggy pass: {fixed_failed_but_buggy_pass_number}")
print(f"Total test generated: {test_generated}")
print(f"Total test successfully compiled: {test_compiled}")
print(f"Total test compiled but failed: {test_failed}")
print(f"Test passed on buggy but failed on fixed: {test_passed_on_buggy_but_failed_on_fixed}")
print(f"Test passed on fixed but failed on buggy: {test_passed_on_fixed_but_failed_on_buggy}")
print(f"Test passed on both: {test_passed_on_both}")
print(f"Test failed on both: {test_failed_on_both}")
print(f"Cannot obtain coverage: {len(cannot_obtain_coverage)}")
if test_generated > 0:
    print(f"Compilable rate: {test_compiled/test_generated}")
    print(f"Runnable to generated rate: {(test_compiled-test_failed)/test_generated}")
else:
    print("Compilable rate not available.")
    print("Runnable to generated rate not available.")
if test_compiled > 0:
    print(f"Runnable to compilable rate: {(test_compiled-test_failed)/test_compiled}")
else:
    print("Runnable to compilable rate not available.")
if total_line_covered + total_line_missed != 0:
    print(f"Total line coverage: {total_line_covered/(total_line_covered+total_line_missed)}")
else:
    print("Total line coverage not available.")
if total_branch_covered + total_branch_missed != 0:
    print(f"Total branch coverage: {total_branch_covered/(total_branch_covered+total_branch_missed)}")
else:
    print("Total branch coverage not available.")
if len(all_line_coverage) != 0:
    print(f"Average line coverage: {np.mean(all_line_coverage)}")
else:
    print("Average line coverage not available.")
if len(all_branch_coverage) != 0:
    print(f"Average branch coverage: {np.mean(all_branch_coverage)}")
else:
    print("Average branch coverage not available.")

In [ ]:
# Save the results to a file
result_file = f'/path/to/repo/data/rq1/results_test/chatgpt/{input_version}_gen_test_results.json'
with open(result_file, 'w') as f:
    total_line_coverage = total_line_covered / (total_line_covered + total_line_missed) * 100 if total_line_covered + total_line_missed != 0 else -1
    total_branch_coverage = total_branch_covered / (total_branch_covered + total_branch_missed) * 100 if total_branch_covered + total_branch_missed != 0 else -1
    avg_line_coverage = np.mean(all_line_coverage) if len(all_line_coverage) != 0 else -1
    avg_branch_coverage = np.mean(all_branch_coverage) if len(all_branch_coverage) != 0 else -1
    results = {
        'total_processed': total_processed,
        'total_defect_processed': len(all_defects),
        'fully_compiled_number': fully_compiled_number,
        'partially_compiled_number': partially_compiled_number,
        'non_compiled_number': non_compiled_number,
        'no_test_generated_number': no_test_generated_number,
        'failed_pre_compile_number': failed_pre_compile_number,
        'compiled_but_run_failed_number': compiled_but_run_failed_number,
        'cannot_obtain_coverage_number': len(cannot_obtain_coverage),
        'bug_detected_number': bug_detected_number,
        'defects_detected_number': len(defects_detected),
        'defects_with_compilable_test_number': len(defect_with_compilable_test),
        'defect_with_invocation_of_focal_method_number': len(focal_method_invoked_defects),
        'buggy_non_compiled_number': buggy_non_compiled_number,
        'fixed_failed_but_buggy_pass_number': fixed_failed_but_buggy_pass_number,
        'test_generated': test_generated,
        'test_compiled': test_compiled,
        'test_failed': test_failed,
        'test_passed_on_buggy_but_failed_on_fixed': test_passed_on_buggy_but_failed_on_fixed,
        'test_passed_on_fixed_but_failed_on_buggy': test_passed_on_fixed_but_failed_on_buggy,
        'test_passed_on_both': test_passed_on_both,
        'test_failed_on_both': test_failed_on_both,
        'compilable_rate': test_compiled/test_generated if test_generated > 0 else -1,
        'runnable_to_generated_rate': (test_compiled-test_failed)/test_generated if test_generated > 0 else -1,
        'runnable_to_compilable_rate': (test_compiled-test_failed)/test_compiled if test_compiled > 0 else -1,
        'total_line_coverage': total_line_coverage,
        'total_branch_coverage': total_branch_coverage,
        'average_line_coverage': avg_line_coverage,
        'average_branch_coverage': avg_branch_coverage,
        'failed_pre_compile': failed_pre_compile,
        'cannot_obtain_coverage': cannot_obtain_coverage,
        'compiled_but_run_failed': compiled_but_run_failed,
        'fully_compiled': fully_compiled,
        'partially_compiled': partially_compiled,
        'non_compiled': non_compiled,
        'no_test_generated': no_test_generated,
        'partially_passed': partially_passed,
        'bug_detected': bug_detected,
        'defects_detected': list(defects_detected),
        'defect_with_compilable_test': list(defect_with_compilable_test),
        'defect_with_invocation_of_focal_method': list(focal_method_invoked_defects),
        'buggy_non_compiled': buggy_non_compiled,
    }
    json.dump(results, f, indent=4)
    print(f"Results saved to {result_file}")